# GOES-16 virtual data ingestion

Tom Nicholas

30th May 2026

## Imports

In [1]:
# std lib
from __future__ import annotations

from collections.abc import Iterator, Iterable
import itertools
import time
from typing import Any
import contextlib
import gc

# platforms
import arraylake as al
import coiled

# standard dependencies
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import datetime
import pandas as pd

# crucial tools for this notebook
import pandera.xarray as pa
import icechunk as ic
#import virtualizarr as vz
import obstore as obs
from obspec_utils.registry import ObjectStoreRegistry

In [2]:
# we want some small unreleased features, should arrive in v2.6.3
!pip install git+https://github.com/zarr-developers/VirtualiZarr.git

In [3]:
import virtualizarr as vz
vz.__version__

'2.6.3.dev3+gc95bc2726'

In [4]:
from virtualizarr.manifests import ManifestArray

In [5]:
hasattr(ManifestArray, "with_fill_value_only")

True

In [6]:
# Icechunk v2.0.5 has some important optimizations for this workload
ic.__version__

'2.0.5'

## Data location

In [7]:
BUCKET = "s3://noaa-goes16"

# For now let's just ingest this product, as it's datacube-like, high-res, full-disk, and has visible bands
PRODUCT = "ABI-L2-MCMIPF/"

In [8]:
# NOAA NODD GOES-R data is on AWS in an anonymous-access bucket in us-east-1 (https://registry.opendata.aws/noaa-goes/)
def _store():
    return obs.store.from_url(BUCKET, region="us-east-1", skip_signature=True)

In [9]:
# GOES-16 went operational in 2017; anything earlier is a sentinel/test directory.
MIN_YEAR = 2017

### Archive split into "Eras" by encoding changes

The compression codecs used for the files change abruptly halfway through the archive. To find the point where the codec changed I had to do a git-bisect-like search through the archive.

It's irritating because this inhomogeneity in the codecs violates the Zarr data model, meaning that the entire timeseries cannot be virtualized as a single Zarr array - the best we can do is two separate arrays, one for all the files with the first codec and one for all the files with the second.

Also, the `scale_factor` and `add_offset` CF encoding values changed twice: once when the satellite ended the pre-operational period and entereed the operational period, and once during the pre-operational period. This causes a similar problem (see [VirtualiZarr issue #1004](https://github.com/zarr-developers/VirtualiZarr/issues/1004) for why), and we end up needing 4 groups.

Note this is also we we don't just concatenate all the different per-band variables into one array. It would be easier to work with (16x fewer variables) but as the different bands use different encoding values it would mean they decode incorrectly.

In [10]:
# =============================================================================
# Era boundaries
# =============================================================================
#
# The archive splits into four era windows driven by three upstream changes at
# NOAA — two calibration transitions and one codec-pipeline transition. Each
# window is internally consistent on *both* dimensions, so an xr.concat-based
# ingest can only mix files from a single window. The v2 ingest writes each
# window into its own zarr group; see REINGEST_PLAN.md.
#
#   pre-2017-07-24   2017-02-28 → 2017-07-24 16:45 UTC
#       chunk codecs: BytesCodec + Zlib (no Shuffle)
#       calibration:  placeholder — C01-C06 use sf=0.000244, ao=0;
#                     C07 uses sf=0.013847, ao=173.15;
#                     C08-C16 all share sf=0.039316, ao=173.15.
#       transition:   2017-07-24 17:15:38.2 UTC — C07-C16 switch to operational
#                     calibration (C01-C06 remain on placeholder). Binary-
#                     searched in debug/bisect_C7C16_boundary.py.
#
#   pre-2017-10-31   2017-07-24 17:15 UTC → 2017-10-31 17:15 UTC
#       chunk codecs: same as above (BytesCodec + Zlib).
#       calibration:  C01-C06 still placeholder (sf=0.000244, ao=0);
#                     C07-C16 now operational per-channel.
#       transition:   2017-10-31 17:30:40.9 UTC — C01-C06 switch to operational.
#                     Binary-searched in debug/bisect_encoding_to_file.py.
#
#   pre-2023-04-19   2017-10-31 17:30 UTC → 2023-04-19 14:50 UTC
#       chunk codecs: same as above.
#       calibration:  fully operational, stable through 2025-04-07
#                     (see debug/dense_encoding_check.py).
#       transition:   2023-04-19 15:00:20.5 UTC — Shuffle filter added to the
#                     codec pipeline.
#
#   post-2023-04-19  2023-04-19 15:00 UTC → present
#       chunk codecs: BytesCodec + Shuffle + Zlib (new Shuffle filter)
#       calibration:  same fully-operational calibration.

# --- late-pre-op boundary (C07-C16 placeholder → operational, 2017-07-24) ---
FIRST_LATE_PREOP_FILE = (
    f"{BUCKET}/ABI-L2-MCMIPF/2017/205/17/"
    "OR_ABI-L2-MCMIPF-M3_G16_s20172051715382_e20172051726154_c20172051726238.nc"
)
LAST_EARLY_PREOP_FILE = (
    f"{BUCKET}/ABI-L2-MCMIPF/2017/205/16/"
    "OR_ABI-L2-MCMIPF-M3_G16_s20172051645382_e20172051656154_c20172051657209.nc"
)
LATE_PREOP_BOUNDARY_DATETIME = np.datetime64("2017-07-24T17:15:38.200", "ns")

# --- operational-calibration boundary (C01-C06 placeholder → operational, 2017-10-31) ---
FIRST_OPERATIONAL_FILE = (
    f"{BUCKET}/ABI-L2-MCMIPF/2017/304/17/"
    "OR_ABI-L2-MCMIPF-M3_G16_s20173041730409_e20173041741176_c20173041741267.nc"
)
LAST_PRE_OPERATIONAL_FILE = (
    f"{BUCKET}/ABI-L2-MCMIPF/2017/304/17/"
    "OR_ABI-L2-MCMIPF-M3_G16_s20173041715409_e20173041726187_c20173041737512.nc"
)
OPERATIONAL_BOUNDARY_DATETIME = np.datetime64("2017-10-31T17:30:40.900", "ns")

# --- codec-change (Shuffle filter) boundary ---
# First file under the new compression codec — pass as `start` to skip the
# older-codec data preceding it.
CODEC_CHANGE_MARKER = (
    f"{BUCKET}/ABI-L2-MCMIPF/2023/109/15/"
    "OR_ABI-L2-MCMIPF-M6_G16_s20231091500205_e20231091509514_c20231091510007.nc"
)
# Last file under the OLD compression codec — pass as `end` to ingest the
# pre-codec-change era (paired with ``start=None``). Scan-start token
# ``20231091450205`` (2023-04-19 14:50 UTC).
LAST_PRE_CODEC_CHANGE_FILE = (
    f"{BUCKET}/ABI-L2-MCMIPF/2023/109/14/"
    "OR_ABI-L2-MCMIPF-M6_G16_s20231091450205_e20231091459527_c20231091500011.nc"
)
CODEC_CHANGE_DATETIME = np.datetime64("2023-04-19T15:00:20.500", "ns")

## Iteration helpers

These functions just help us iterate over the files as they are laid out in the bucket.

In [11]:
def _parse_url_to_day(url: str) -> tuple[int, int]:
    """Extract ``(year, doy)`` from a GOES file URL or key."""
    parts = url.split("/")
    try:
        idx = parts.index("ABI-L2-MCMIPF")
    except ValueError as e:
        raise ValueError(f"URL doesn't contain ABI-L2-MCMIPF: {url}") from e
    if len(parts) < idx + 3:
        raise ValueError(f"URL doesn't have year/doy: {url}")
    return int(parts[idx + 1]), int(parts[idx + 2])


def _common_prefix_children(prefix: str) -> list[str]:
    store = _store()
    result = obs.list_with_delimiter(store, prefix=prefix)
    return sorted(
        p[len(prefix):].rstrip("/") for p in result["common_prefixes"]
    )


def list_years() -> list[int]:
    """Every year directory present under the product prefix, filtered to ``>= MIN_YEAR``.

    Defensive against listings that include non-year entries: strips slashes
    from each child (different obstore versions are inconsistent about whether
    they preserve trailing slashes on common_prefixes), then requires a strict
    4-digit numeric token before attempting ``int()``. The noaa-goes16 bucket
    has been observed to contain at least one pre-2017 sentinel directory
    (``ABI-L2-MCMIPF/2000/``) which our existing ``>= MIN_YEAR`` filter would
    drop — but only if the int conversion succeeds first.
    """
    out: list[int] = []
    for c in _common_prefix_children(PRODUCT):
        c = c.strip("/")
        if len(c) == 4 and c.isdigit() and int(c) >= MIN_YEAR:
            out.append(int(c))
    return sorted(out)


def list_days_in_year(year: int) -> list[int]:
    """Every DOY directory present in a given year.

    Defensive against non-DOY entries for the same reason as :func:`list_years`.
    """
    out: list[int] = []
    for c in _common_prefix_children(f"{PRODUCT}{year}/"):
        c = c.strip("/")
        if len(c) == 3 and c.isdigit():
            out.append(int(c))
    return sorted(out)


def iter_days(
    *,
    start: str | None = None,
    end: str | None = None,
) -> Iterator[tuple[int, int]]:
    """Yield ``(year, doy)`` for every day in the archive, in chronological order.

    Days are inclusively clipped to those at or after ``start`` and at or before
    ``end``. ``start``/``end`` are full file URLs (or anything compatible with
    :func:`_parse_url_to_day`).
    """
    start_day = _parse_url_to_day(start) if start else None
    end_day = _parse_url_to_day(end) if end else None

    for year in list_years():
        if start_day and year < start_day[0]:
            continue
        if end_day and year > end_day[0]:
            return
        for doy in list_days_in_year(year):
            current = (year, doy)
            if start_day and current < start_day:
                continue
            if end_day and current > end_day:
                return
            yield current


def _is_aborted_scan(path: str) -> bool:
    """True if the filename's ``s`` (scan-start) and ``e`` (scan-end) tokens are
    identical — a signature of an aborted/placeholder file with a sentinel ``t``
    coordinate that would corrupt downstream concatenation."""
    fname = path.rsplit("/", 1)[-1]
    try:
        s = fname.split("_s")[1].split("_")[0]
        e = fname.split("_e")[1].split("_")[0]
    except IndexError:
        return False
    return s == e


def _scan_start_token(url: str) -> str:
    """Extract the ``sYYYYDDDHHMMSST`` token from a GOES filename.

    Sorting by this string gives chronological order independent of the scan-
    mode token in the filename (``M3``/``M4``/``M6``), which a plain lex sort
    of the whole URL would otherwise interleave incorrectly (e.g. ``M4`` <
    ``M6`` lex-sorts an M4 file at 10:50 before all M6 files in the same hour).
    """
    fname = url.rsplit("/", 1)[-1]
    return fname.split("_s")[1].split("_")[0]


def parse_scan_start_to_datetime(url_or_filename: str) -> np.datetime64:
    """Parse the ``sYYYYDDDHHMMSST`` filename token into a ``datetime64[ns]``.

    Example: ``"...s20231091500205..."`` →
    ``np.datetime64("2023-04-19T15:00:20.500")``.
    """
    token = _scan_start_token(url_or_filename)
    if len(token) != 14:
        raise ValueError(f"Unexpected scan-start token length: {token!r}")
    year = int(token[0:4])
    doy = int(token[4:7])
    hour = int(token[7:9])
    minute = int(token[9:11])
    sec = int(token[11:13])
    tenths = int(token[13:14])
    date = datetime.date(year, 1, 1) + datetime.timedelta(days=doy - 1)
    return np.datetime64(
        datetime.datetime(date.year, date.month, date.day, hour, minute, sec, tenths * 100_000),
        "ns",
    )


def list_day_files(
    year: int,
    doy: int,
    *,
    start: str | None = None,
    end: str | None = None,
) -> list[str]:
    """Every operational ``.nc`` file URL in a single day, in chronological scan-time order.

    Files where the scan-start (``_s``) and scan-end (``_e``) tokens in the
    filename are identical are filtered out — those are aborted/placeholder
    scans with sentinel ``t = J2000 epoch`` values that corrupt downstream
    combining. Both ``M6`` (operational 10-minute flex) and ``M4`` (continuous
    5-minute full disk, used during special operations) are kept; they share
    the same schema and codecs.

    URLs are inclusively clipped to those at or after ``start`` and at or before
    ``end``. Lex sort of the full URL matches scan-time order (the filename
    starts ``OR_..._sYYYYDDDHHMMSST...``).
    """
    store = _store()
    day_prefix = f"{PRODUCT}{year}/{doy:03d}/"
    urls: list[str] = []
    for page in obs.list(store, prefix=day_prefix):
        for obj in page:
            path = obj["path"]
            if path.endswith(".nc") and not _is_aborted_scan(path):
                urls.append(f"{BUCKET}/{path}")
    urls.sort(key=_scan_start_token)

    # Deduplicate by scan-start token: some NOAA hour prefixes get listed
    # under both `.../H/` and `.../HH/` (single-digit and zero-padded), so
    # obstore returns the same underlying file twice. Keep the first occurrence.
    seen: set[str] = set()
    deduped: list[str] = []
    for u in urls:
        tok = _scan_start_token(u)
        if tok not in seen:
            seen.add(tok)
            deduped.append(u)
    urls = deduped

    if start is not None:
        urls = [u for u in urls if _scan_start_token(u) >= _scan_start_token(start)]
    if end is not None:
        urls = [u for u in urls if _scan_start_token(u) <= _scan_start_token(end)]
    return urls


def iter_archive(
    *,
    start: str | None = None,
    end: str | None = None,
) -> Iterator[str]:
    """Flat iterator over every file URL across the archive, clipped to ``[start, end]``."""
    for year, doy in iter_days(start=start, end=end):
        yield from list_day_files(year, doy, start=start, end=end)


def iter_archive_by_day(
    *,
    start: str | None = None,
    end: str | None = None,
) -> Iterator[tuple[tuple[int, int], list[str]]]:
    """Per-day iterator: yield ``((year, doy), [urls])`` for each day in order.

    Use this when you want to combine and commit one day's worth of files before
    moving on to the next. ``start``/``end`` are applied consistently to both
    the outer day iteration and the inner file list, so the first and last day
    will already be clipped to the requested range.
    """
    for year, doy in iter_days(start=start, end=end):
        yield (year, doy), list_day_files(year, doy, start=start, end=end)

## Preprocessing pipeline

The GOES-16 data in its raw form requires validation for consistency (e.g. verifying codecs are as expected), some cleaning and reorganizing to make a nicer datacube, then validation of the cleaned result.

We use `pandera.xarray` ([docs](https://pandera.readthedocs.io/en/v0.31.1/xarray_guide/index.html)) to help with defining the expected schema for the files and validating their contents against that schema. Doing this validation up front, before concatenation, makes it easier to debug if an inconsistency is encountered. We also take advantage of pandera's support for duck-typed arrays to perform this validation on the virtual dataset object directly, so we only need to parse the file once, and we can perform the validation as part of the `preprocess` kwarg to `open_virtual_mfdataset`.

One of the main things to clean up is that the data holds each band in a separate variable, when they can more naturally be thought of as stackable along a common `band` dimension. However some bands appear to be missing from some files, so we fill those locations in the array with NaNs.

The pipeline is a series of chainable ``xr.Dataset -> xr.Dataset`` functions:

    validate_raw             — pandera structural validation + codec validation
                              of the raw virtualizarr-opened file. Pass-through.
    fill_missing_channels    — back-fill any missing per-channel variable.
    stack_channels           — concat ``CMI_C*``/``DQF_C*``/per-channel stats
                              along a new ``band`` dim. Builds ``band`` and
                              ``wavelength`` coords.
    add_time_dimension       — ``expand_dims('t')`` on every data variable using
                              the existing scalar ``t`` coord.
    validate_cleaned         — pandera structural validation of the cleaned dataset.

`preprocess` chains them all and raises immediately on the first failure.
Pandera's ``lazy=True`` still collects every schema-level error within a single validation call, so the report from whichever stage fails first shows every issue it found. 
The raised `MCMIPFValidationError` attaches the source filename (from ``ds.encoding['source']`` or ``ds.attrs['dataset_name']``).

## Calibration validation

Since the calibrations changed and have to be consistent across files for the virtual chunks to decode correctly (see above), we double-check that each file has the expected calibration upon ingestion.

In [12]:
"""Per-channel CMI calibration tables and per-file calibration validation.

The GOES-16 ABI-L2-MCMIPF archive went through two upstream calibration
transitions before settling on the operational per-channel values:

  1. ``2017-07-24T17:15:38.2 UTC`` — C07-C16 (brightness temperature) switched
     from placeholder to operational. C01-C06 (reflectance) still on placeholder.
     Bisected by ``debug/bisect_C7C16_boundary.py``.

  2. ``2017-10-31T17:30:40.9 UTC`` — C01-C06 (reflectance) switched to
     operational. Bisected by ``debug/bisect_encoding_to_file.py``.

That gives three calibration eras, each with a fixed 16-channel
``scale_factor`` / ``add_offset`` table. Files are routed into one of four
ingest groups by the combination of calibration era and codec era; see
``REINGEST_PLAN.md``.

This module is the single source of truth for the per-era calibration tables
and provides:

  * :data:`EARLY_PREOP_CALIBRATION`, :data:`LATE_PREOP_CALIBRATION`,
    :data:`OPERATIONAL_CALIBRATION` — per-channel ``(scale_factor, add_offset)``
    tables, harvested directly from boundary files via h5py and stored as
    ``np.float32`` so callers can validate with ``np.isclose`` against the
    JSON-promoted float64 values surfaced by VirtualiZarr's HDFParser.

  * :func:`calibration_era_for_ds` — pick the right table based on a dataset's
    scalar ``t``.

  * :func:`check_calibration` — assert that the dataset's per-channel attrs
    match the expected table. Raises :class:`CalibrationMismatchError` on any
    deviation; intended to be called from ``validate_raw`` so a file with
    unexpected calibration (e.g. an unknown sub-era we missed during
    exploration) gets rejected before being mixed into a batch.
"""

from typing import Literal

CalibrationEra = Literal["early-preop", "late-preop", "operational"]
_ChannelCal = dict[str, dict[str, np.float32]]


# =============================================================================
# Per-era calibration tables
# =============================================================================
#
# Values are stored as np.float32 because that's the source NetCDF's binary
# storage dtype. JSON serialization (zarr v3, VirtualiZarr's HDFParser, etc.)
# promotes them to Python float (float64) when round-tripping; check_calibration
# below uses np.isclose to tolerate that drift.
#
# Harvest scripts: /tmp/harvest_calibration.py (operational) and
# /tmp/harvest_early.py (early pre-op). Source files documented inline.

# Harvested from FIRST_OPERATIONAL_FILE (s20173041730409, 2017-10-31 17:30 UTC).
# Confirmed stable through 2025-04-07 by debug/dense_encoding_check.py.
OPERATIONAL_CALIBRATION: _ChannelCal = {
    "CMI_C01": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C02": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C03": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C04": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C05": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C06": {"scale_factor": np.float32(0.00031746001332066953), "add_offset": np.float32(0.0)},
    "CMI_C07": {"scale_factor": np.float32(0.013096179813146591),   "add_offset": np.float32(197.30999755859375)},
    "CMI_C08": {"scale_factor": np.float32(0.042249858379364014),   "add_offset": np.float32(138.0500030517578)},
    "CMI_C09": {"scale_factor": np.float32(0.042339108884334564),   "add_offset": np.float32(137.6999969482422)},
    "CMI_C10": {"scale_factor": np.float32(0.04988918825984001),    "add_offset": np.float32(126.91000366210938)},
    "CMI_C11": {"scale_factor": np.float32(0.05216431990265846),    "add_offset": np.float32(127.69000244140625)},
    "CMI_C12": {"scale_factor": np.float32(0.047270338982343674),   "add_offset": np.float32(117.48999786376953)},
    "CMI_C13": {"scale_factor": np.float32(0.06145332008600235),    "add_offset": np.float32(89.62000274658203)},
    "CMI_C14": {"scale_factor": np.float32(0.059850748628377914),   "add_offset": np.float32(96.19000244140625)},
    "CMI_C15": {"scale_factor": np.float32(0.05956082046031952),    "add_offset": np.float32(97.37999725341797)},
    "CMI_C16": {"scale_factor": np.float32(0.055081531405448914),   "add_offset": np.float32(92.69999694824219)},
}

# Harvested from LAST_EARLY_PREOP_FILE (s20172051645382, 2017-07-24 16:45 UTC).
# C01-C06 reflectance: uniform placeholder (sf=0.000244, ao=0).
# C07: unique brightness placeholder. C08-C16: shared brightness placeholder.
EARLY_PREOP_CALIBRATION: _ChannelCal = {
    "CMI_C01": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C02": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C03": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C04": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C05": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C06": {"scale_factor": np.float32(0.00024419999681413174), "add_offset": np.float32(0.0)},
    "CMI_C07": {"scale_factor": np.float32(0.013846670277416706),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C08": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C09": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C10": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C11": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C12": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C13": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C14": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C15": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
    "CMI_C16": {"scale_factor": np.float32(0.039316240698099136),   "add_offset": np.float32(173.14999389648438)},
}

# Built from above: C01-C06 still on early-pre-op placeholder; C07-C16 already
# on operational values. Spot-verified against LAST_PRE_OPERATIONAL_FILE
# (s20173041715409).
LATE_PREOP_CALIBRATION: _ChannelCal = {
    "CMI_C01": EARLY_PREOP_CALIBRATION["CMI_C01"],
    "CMI_C02": EARLY_PREOP_CALIBRATION["CMI_C02"],
    "CMI_C03": EARLY_PREOP_CALIBRATION["CMI_C03"],
    "CMI_C04": EARLY_PREOP_CALIBRATION["CMI_C04"],
    "CMI_C05": EARLY_PREOP_CALIBRATION["CMI_C05"],
    "CMI_C06": EARLY_PREOP_CALIBRATION["CMI_C06"],
    "CMI_C07": OPERATIONAL_CALIBRATION["CMI_C07"],
    "CMI_C08": OPERATIONAL_CALIBRATION["CMI_C08"],
    "CMI_C09": OPERATIONAL_CALIBRATION["CMI_C09"],
    "CMI_C10": OPERATIONAL_CALIBRATION["CMI_C10"],
    "CMI_C11": OPERATIONAL_CALIBRATION["CMI_C11"],
    "CMI_C12": OPERATIONAL_CALIBRATION["CMI_C12"],
    "CMI_C13": OPERATIONAL_CALIBRATION["CMI_C13"],
    "CMI_C14": OPERATIONAL_CALIBRATION["CMI_C14"],
    "CMI_C15": OPERATIONAL_CALIBRATION["CMI_C15"],
    "CMI_C16": OPERATIONAL_CALIBRATION["CMI_C16"],
}


_CALIBRATION_BY_ERA: dict[CalibrationEra, _ChannelCal] = {
    "early-preop": EARLY_PREOP_CALIBRATION,
    "late-preop":  LATE_PREOP_CALIBRATION,
    "operational": OPERATIONAL_CALIBRATION,
}


# =============================================================================
# Era classification
# =============================================================================

def calibration_era_for_ds(ds: xr.Dataset) -> CalibrationEra:
    """Return which calibration era a single-file dataset belongs to.

    Uses the scalar ``t`` coordinate to place the file relative to the two
    calibration boundaries (LATE_PREOP and OPERATIONAL). Raises if ``t`` is
    not a scalar (i.e. the dataset already has a ``t`` dim from concat).
    """
    if "t" not in ds.coords:
        raise ValueError("Dataset has no 't' coord — cannot classify calibration era.")
    t_arr = np.asarray(ds["t"].values)
    if t_arr.ndim != 0:
        raise ValueError(
            f"Expected a scalar 't' (single-file ds), got shape {t_arr.shape}. "
            "Call calibration_era_for_ds per file, before any concat."
        )
    t = np.datetime64(t_arr, "ns")
    if t >= OPERATIONAL_BOUNDARY_DATETIME:
        return "operational"
    if t >= LATE_PREOP_BOUNDARY_DATETIME:
        return "late-preop"
    return "early-preop"


# =============================================================================
# Validation
# =============================================================================

class CalibrationMismatchError(ValueError):
    """Raised when a file's CMI_C{NN} calibration doesn't match the expected era."""

    def __init__(self, era: CalibrationEra, mismatches: list[tuple[str, str, Any, Any]]):
        self.era = era
        self.mismatches = mismatches
        lines = [
            f"CMI calibration doesn't match expected era={era!r} "
            f"({len(mismatches)} mismatch(es)):",
        ]
        for name, key, actual, expected in mismatches:
            lines.append(f"  {name}.attrs[{key!r}] = {actual!r}, expected {expected!r}")
        lines.append(
            "If this is a previously-undiscovered sub-era, harvest its calibration "
            "table and add a new entry alongside EARLY_PREOP_CALIBRATION etc."
        )
        super().__init__("\n".join(lines))


def check_calibration(ds: xr.Dataset) -> None:
    """Validate that every present ``CMI_C{NN}``'s scale_factor / add_offset
    matches the expected era's table.

    The dataset's era is inferred from its scalar ``t`` via
    :func:`calibration_era_for_ds`. Missing channels are ignored — they'll be
    synthesized by ``fill_missing_channels``. Comparison uses ``np.isclose``
    with default tolerances; this absorbs the float32 → float64 JSON
    promotion that VirtualiZarr's HDFParser introduces (REINGEST_PLAN.md
    issue 4a).

    Raises :class:`CalibrationMismatchError` if any channel disagrees,
    accumulating every mismatch in a single error so the report is complete.
    """
    era = calibration_era_for_ds(ds)
    expected_table = _CALIBRATION_BY_ERA[era]

    mismatches: list[tuple[str, str, Any, Any]] = []
    for i in range(1, 17):
        name = f"CMI_C{i:02d}"
        if name not in ds.variables:
            continue
        attrs = ds[name].attrs
        expected = expected_table[name]
        for key in ("scale_factor", "add_offset"):
            actual = attrs.get(key)
            expected_val = expected[key]
            if actual is None:
                mismatches.append((name, key, "MISSING", expected_val))
                continue
            if not np.isclose(actual, expected_val):
                mismatches.append((name, key, actual, expected_val))

    if mismatches:
        raise CalibrationMismatchError(era, mismatches)


## Data value verifier

As an additional sanity check we use this script (or the functions within) to compare the actual values read and decoded from a source netCDF file vs the resultant Icechunk store. It would be expensive to do this check on every file so we will only perform it on a subset.

In [13]:
#!/usr/bin/env -S uv run
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "arraylake>=0.28",
#     "icechunk>=2.0",
#     "xarray",
#     "zarr",
#     "numpy",
#     "dask",
#     "h5netcdf",
#     "h5py",
#     "obstore",
#     "obspec-utils",
# ]
# ///
"""Compare a single committed scan in icechunk against its source NetCDF.

The point of this module is to catch defects in the ingest pipeline that
*decode* differently from the original — wrong scale_factor on a chunk,
clobbered CF attrs, swapped channels, off-by-one chunk references, etc.

Workflow: given a target ``t`` (or a source URL), open the same scan
from both sides — the committed icechunk dataset (a slice at the matching
``t``) and the original NetCDF on S3 (downloaded to a temp file) — fully
CF-decoded by xarray on both sides, then assert exact equality on every
variable they share. Raises :class:`VerificationError` with a per-variable
mismatch report if anything disagrees.

Intended for both ad-hoc spot-checks and post-batch verification inside
``ingest_all_days``.

    # Ad-hoc
    verify_timestep(
        repo=repo, branch="v2-ingest",
        group="ABI-L2-MCMIPF/post-2023-04-19",
        t=np.datetime64("2023-04-19T15:00:20.500"),
    )

The function is intentionally chatty (prints a one-line summary on
success) so it's useful as a standalone CLI.
"""

from __future__ import annotations

import contextlib
import datetime
import os
import sys
import tempfile
import time
from collections.abc import Sequence
from dataclasses import dataclass, field
from typing import Any

import numpy as np
import obstore as obs
import xarray as xr
from obspec_utils.registry import ObjectStoreRegistry


# =============================================================================
# Constants
# =============================================================================

# Attrs that xarray / icechunk add or strip on round-trip and that we don't
# expect to match between a freshly-decoded NetCDF and the icechunk read.
# Anything not in this set is required to match exactly.
_IGNORED_ATTRS: frozenset[str] = frozenset({
    # xarray internal — present after CF decoding, absent on the wire
    "_FillValue",
    # Zarr v3 stores chunk encoding outside the variable attrs; the source
    # NetCDF has these in attrs, icechunk surfaces them via .encoding instead.
    "scale_factor",
    "add_offset",
    # Often set by xarray's decoder during CF decoding
    "coordinates",
    # Zarr adds this
    "_ARRAY_DIMENSIONS",
})

# Attrs that are *expected* on per-channel data vars and which we DO want to
# compare exactly. (We deliberately re-include scale_factor/add_offset/_FillValue
# here as encoding, not attrs; see _compare_encoding below.)
_ATTRS_OF_INTEREST: tuple[str, ...] = (
    "long_name",
    "standard_name",
    "units",
    "valid_range",
    "grid_mapping",
    "ancillary_variables",
    "resolution",
    "cell_methods",
    "axis",
)

# Encoding keys that affect physical decoding and must agree exactly.
_ENCODING_OF_INTEREST: tuple[str, ...] = (
    "scale_factor",
    "add_offset",
    "_FillValue",
    "dtype",
)


# =============================================================================
# Result types
# =============================================================================

@dataclass
class FieldMismatch:
    name: str
    kind: str           # "values" | "shape" | "dtype" | "attr" | "encoding" | "missing"
    detail: str

    def __str__(self) -> str:
        return f"  [{self.kind}] {self.name}: {self.detail}"


class VerificationError(AssertionError):
    """Raised when verify_timestep finds disagreements between icechunk and source."""

    def __init__(
        self,
        t: np.datetime64,
        source_url: str,
        group: str,
        mismatches: Sequence[FieldMismatch],
    ):
        self.t = t
        self.source_url = source_url
        self.group = group
        self.mismatches = list(mismatches)
        lines = [
            f"VerificationError: {len(mismatches)} mismatch(es) at t={t} in {group}",
            f"  source: {source_url}",
            *[str(m) for m in mismatches],
        ]
        super().__init__("\n".join(lines))


@dataclass
class VerificationReport:
    """Returned on success — summary of what was compared."""
    t: np.datetime64
    source_url: str
    group: str
    variables_compared: list[str] = field(default_factory=list)
    variables_only_in_icechunk: list[str] = field(default_factory=list)
    variables_only_in_source: list[str] = field(default_factory=list)

    def __str__(self) -> str:
        return (
            f"VerificationReport(t={self.t}, group={self.group!r}, "
            f"compared {len(self.variables_compared)} vars; "
            f"only_in_icechunk={len(self.variables_only_in_icechunk)}, "
            f"only_in_source={len(self.variables_only_in_source)})"
        )


# =============================================================================
# Resolution / fetching
# =============================================================================

def _ensure_datetime64_ns(t: Any) -> np.datetime64:
    if isinstance(t, np.datetime64):
        return t.astype("datetime64[ns]")
    return np.datetime64(t, "ns")


# The filename `_s_` token in GOES MCMIPF products is the scan-command time,
# but the `t` variable stored inside the NetCDF is the mid-scan time — they
# differ by roughly the scan duration. We don't know the exact offset for an
# arbitrary file ahead of time, so we widen the candidate window to be safe
# and fall back on inspecting each file's actual `t` to confirm the match.
_FILENAME_TOKEN_TO_T_OFFSET_WINDOW = datetime.timedelta(minutes=20)


def _read_t_from_netcdf(path: str) -> np.datetime64:
    """Open a downloaded source NetCDF and return its scalar ``t`` (mid-scan time)."""
    with xr.open_dataset(path, engine="h5netcdf") as ds:
        return _ensure_datetime64_ns(ds["t"].values)


def _t_to_source_url(t: np.datetime64, *, registry: ObjectStoreRegistry | None = None) -> str:
    """Locate the source NetCDF whose internal ``t`` equals the requested ``t``.

    The filename ``_s_`` token is a coarse approximation of ``t`` (it's the
    scan-command time, which precedes the mid-scan ``t`` by roughly the scan
    duration). So this:

      1. lists the day's files,
      2. selects candidates whose filename-token-time falls within
         ``_FILENAME_TOKEN_TO_T_OFFSET_WINDOW`` before the requested ``t``,
      3. fetches each candidate and inspects its actual ``t``,
      4. returns the URL whose internal ``t`` matches exactly.
    """
    ts = _ensure_datetime64_ns(t)
    py_dt = ts.astype("datetime64[us]").astype(datetime.datetime)

    # Generate candidate URLs from this day and the immediately preceding
    # day (in case ts is just past midnight).
    candidates: list[str] = []
    for offset_days in (-1, 0):
        date = (py_dt + datetime.timedelta(days=offset_days)).date()
        doy = date.timetuple().tm_yday
        try:
            candidates.extend(list_day_files(date.year, doy))
        except Exception:
            pass

    window_start = py_dt - _FILENAME_TOKEN_TO_T_OFFSET_WINDOW
    window_end = py_dt
    filtered: list[str] = []
    for url in candidates:
        token_t = parse_scan_start_to_datetime(url).astype("datetime64[us]").astype(datetime.datetime)
        if window_start <= token_t <= window_end:
            filtered.append(url)

    if not filtered:
        raise FileNotFoundError(
            f"No candidate NetCDF in {py_dt.date()} whose filename-token-time "
            f"falls within {_FILENAME_TOKEN_TO_T_OFFSET_WINDOW} before t={t}"
        )

    for url in filtered:
        tmp = _fetch_source_to_tempfile(url, registry=registry)
        try:
            file_t = _read_t_from_netcdf(tmp)
            if file_t == ts:
                return url
        finally:
            try:
                os.unlink(tmp)
            except OSError:
                pass

    raise FileNotFoundError(
        f"None of {len(filtered)} candidate file(s) near t={t} had a matching "
        f"internal t value. Candidates tried: {[u.rsplit('/', 1)[-1] for u in filtered]}"
    )


def _fetch_source_to_tempfile(url: str, *, registry: ObjectStoreRegistry | None = None) -> str:
    """Download a single S3 NetCDF to a temp path; return the path."""
    if not url.startswith(BUCKET + "/"):
        raise ValueError(f"URL must live under {BUCKET}; got {url!r}")
    key = url[len(BUCKET) + 1:]

    if registry is not None:
        store, _ = registry.resolve(url)
    else:
        store = obs.store.from_url(BUCKET, region="us-east-1", skip_signature=True)

    blob = obs.get(store, key).bytes().to_bytes()
    fd, path = tempfile.mkstemp(suffix=".nc", prefix="mcmipf_verify_")
    with os.fdopen(fd, "wb") as f:
        f.write(blob)
    return path


# =============================================================================
# Comparison primitives
# =============================================================================

def _values_equal(a: np.ndarray, b: np.ndarray) -> tuple[bool, str]:
    """Compare arrays for equality, NaN-aware.

    For float-dtype operands uses :func:`numpy.isclose` with the default
    tolerances. This tolerates the precision drift that Zarr v3's JSON
    metadata format introduces: a float32 ``scale_factor`` written to disk
    is read back as a Python ``float`` (float64), so the decoded result is
    float64 with float32-equivalent values padded by zero LSBs. Same goes
    for ``x`` / ``y`` and any other variable that round-trips through CF
    scale_factor / add_offset.

    The proper fix lives upstream — moving CF scale/offset decoding into
    a Zarr v3 codec (see REINGEST_PLAN.md issue 4a) — and would make exact
    comparison hold. Until then ``allclose``-style is the closest signal
    we can get without false alarms.

    Non-float operands stay strictly exact-equal.
    """
    if a.shape != b.shape:
        return False, f"shape {a.shape} vs {b.shape}"
    if np.issubdtype(a.dtype, np.floating) or np.issubdtype(b.dtype, np.floating):
        close = np.isclose(a, b, equal_nan=True)
        ok = bool(close.all())
    else:
        close = a == b
        ok = bool(close.all())
    if ok:
        return True, ""
    diff = ~close
    n_diff = int(diff.sum())
    idx = np.argwhere(diff)
    first = tuple(int(v) for v in idx[0]) if idx.size else ()
    a_v = a[first] if first else a
    b_v = b[first] if first else b
    return False, (
        f"{n_diff} pixels differ (of {a.size}); first @ {first}: "
        f"icechunk={a_v!r} source={b_v!r}"
    )


def _valid_range_is_superset(ic_v: Any, src_v: Any) -> bool:
    """True iff ``ic_v`` (a ``[min, max]`` pair) fully contains ``src_v``.

    NOAA has monotonically expanded ``DQF_C{NN}.valid_range`` over time —
    e.g. from ``[0, 3]`` (early-2017 source) to ``[0, 4]`` (2019+ source)
    when a new DQF flag value was added. The pipeline canonicalises the
    icechunk-stored value to the wider modern range; this check accepts
    older narrower source ranges as a benign subset.

    A narrower icechunk range than source's would still fail — that would
    indicate the icechunk store is misrepresenting the data's valid range.
    """
    try:
        ic_arr = np.asarray(ic_v)
        src_arr = np.asarray(src_v)
    except Exception:
        return False
    if ic_arr.shape != (2,) or src_arr.shape != (2,):
        return False
    return bool(ic_arr[0] <= src_arr[0] and ic_arr[1] >= src_arr[1])


def _compare_attrs(name: str, ic: xr.DataArray, src: xr.DataArray) -> list[FieldMismatch]:
    mismatches: list[FieldMismatch] = []
    for key in _ATTRS_OF_INTEREST:
        ic_has = key in ic.attrs
        src_has = key in src.attrs
        if not ic_has and not src_has:
            continue
        if ic_has != src_has:
            mismatches.append(FieldMismatch(
                f"{name}.attrs[{key!r}]", "attr",
                f"present in {'icechunk' if ic_has else 'source'} only",
            ))
            continue
        ic_v, src_v = ic.attrs[key], src.attrs[key]

        if key == "valid_range":
            # Accept icechunk range as a superset of source's. See
            # _valid_range_is_superset docstring + GOES_DATA_INCONSISTENCIES.md.
            if not _valid_range_is_superset(ic_v, src_v):
                mismatches.append(FieldMismatch(
                    f"{name}.attrs[{key!r}]", "attr",
                    f"icechunk valid_range {list(np.asarray(ic_v))!r} does not "
                    f"contain source valid_range {list(np.asarray(src_v))!r}",
                ))
            continue

        if not _scalar_or_array_equal(ic_v, src_v):
            mismatches.append(FieldMismatch(
                f"{name}.attrs[{key!r}]", "attr",
                f"icechunk={ic_v!r} source={src_v!r}",
            ))
    return mismatches


def _compare_encoding(name: str, ic: xr.DataArray, src: xr.DataArray) -> list[FieldMismatch]:
    """Compare the CF encoding keys that affect physical decoding."""
    mismatches: list[FieldMismatch] = []
    for key in _ENCODING_OF_INTEREST:
        ic_v = ic.encoding.get(key, ic.attrs.get(key))
        src_v = src.encoding.get(key, src.attrs.get(key))
        if ic_v is None and src_v is None:
            continue
        if (ic_v is None) != (src_v is None):
            mismatches.append(FieldMismatch(
                f"{name}.encoding[{key!r}]", "encoding",
                f"present in {'icechunk' if ic_v is not None else 'source'} only "
                f"(icechunk={ic_v!r}, source={src_v!r})",
            ))
            continue
        if key == "dtype":
            if np.dtype(ic_v) != np.dtype(src_v):
                mismatches.append(FieldMismatch(
                    f"{name}.encoding[{key!r}]", "encoding",
                    f"icechunk={ic_v} source={src_v}",
                ))
            continue
        if not _scalar_or_array_equal(ic_v, src_v):
            mismatches.append(FieldMismatch(
                f"{name}.encoding[{key!r}]", "encoding",
                f"icechunk={ic_v!r} source={src_v!r}",
            ))
    return mismatches


def _scalar_or_array_equal(a: Any, b: Any) -> bool:
    """Compare two values that may be scalars, numpy arrays, lists, or strings.

    Float-kind operands go through :func:`numpy.allclose` (NaN-aware) rather
    than exact equality, for the same JSON-promotion reason documented on
    :func:`_values_equal`. Applies in particular to ``scale_factor`` and
    ``add_offset`` attr/encoding comparisons.
    """
    try:
        aa = np.asarray(a)
        bb = np.asarray(b)
    except Exception:
        return a == b
    if aa.shape != bb.shape:
        return False
    if aa.dtype.kind in "fc" or bb.dtype.kind in "fc":
        try:
            return bool(np.allclose(aa, bb, equal_nan=True))
        except TypeError:
            return bool(np.allclose(aa, bb))
    if aa.dtype.kind in "US" or bb.dtype.kind in "US":
        return bool((aa == bb).all())
    return bool(np.array_equal(aa, bb))


def _compare_variable(
    name: str, ic_var: xr.DataArray, src_var: xr.DataArray,
) -> list[FieldMismatch]:
    """Compare values + relevant attrs/encoding of one variable across both sides."""
    mismatches: list[FieldMismatch] = []

    # Strip the icechunk's t dim if present (we should already have isel'd it,
    # but be defensive in case a (1,) leftover exists).
    ic_a = ic_var.squeeze(drop=True) if "t" in ic_var.dims else ic_var
    src_a = src_var.squeeze(drop=True) if "t" in src_var.dims else src_var

    ic_vals = np.asarray(ic_a.values)
    src_vals = np.asarray(src_a.values)

    # Treat float32 vs float64 as a non-defect: it's an artefact of CF
    # scale_factor/add_offset round-tripping through Zarr v3 JSON metadata
    # (see REINGEST_PLAN.md issue 4a). Value-level agreement is still
    # enforced by _values_equal's allclose check. Any other dtype mismatch
    # (e.g. float vs int, or int widths differing) remains a hard failure.
    both_float = (
        np.issubdtype(ic_vals.dtype, np.floating)
        and np.issubdtype(src_vals.dtype, np.floating)
    )
    if ic_vals.dtype != src_vals.dtype and not both_float:
        mismatches.append(FieldMismatch(
            name, "dtype",
            f"icechunk={ic_vals.dtype} source={src_vals.dtype}",
        ))

    ok, detail = _values_equal(ic_vals, src_vals)
    if not ok:
        mismatches.append(FieldMismatch(name, "values", detail))

    mismatches.extend(_compare_attrs(name, ic_a, src_a))
    mismatches.extend(_compare_encoding(name, ic_a, src_a))
    return mismatches


# =============================================================================
# Main entry point
# =============================================================================

def verify_timestep(
    *,
    repo,
    group: str,
    branch: str = "main",
    t: Any = None,
    source_url: str | None = None,
    registry: ObjectStoreRegistry | None = None,
    variables: Sequence[str] | None = None,
    skip_variables: Sequence[str] = (),
    verbose: bool = True,
    zarr_async_concurrency: int | None = None,
    force_synchronous: bool = False,
) -> VerificationReport:
    """Compare a single scan between the icechunk repo and the original NetCDF.

    Exactly one of ``t`` or ``source_url`` should be provided (if both, they must
    agree). Raises :class:`VerificationError` on any disagreement.

    Parameters
    ----------
    repo : arraylake / icechunk Repository
    group : str
        Zarr group path within the repo (e.g. ``"ABI-L2-MCMIPF/post-2023-04-19"``).
    branch : str
        Branch to read from (default ``"main"``).
    t, source_url : scan identifier
        One of these is required.
    registry : ObjectStoreRegistry, optional
        Reused obstore registry for the NOAA bucket. Created on the fly if absent.
    variables : sequence of str, optional
        If given, only compare these variables.
    skip_variables : sequence of str
        Variables present on both sides to skip from value comparison.
    verbose : bool
        Print a one-line summary on success.
    zarr_async_concurrency : int, optional
        If set, calls ``zarr.config.set({"async.concurrency": N})`` at the
        start of the call so the icechunk-side chunk reads can fan out wider.
        Recommended values:
            16 — small / slow-connection machines
            64 — high-performance cloud workers
        Defaults to ``None`` (no change to global zarr config). Note this
        mutates zarr's process-global config; when running under dask-delayed
        on workers, each worker picks it up on its first call.
    force_synchronous : bool
        If True, open the icechunk side with ``chunks=None`` so the chunk
        reads run synchronously on the current process (no dask dispatch).
        Use this as a fallback when the default cluster-dispatched path
        fails due to stale credentials on workers — the current process
        (driver) has a fresh repo handle, so the synchronous path bypasses
        the worker credential issue entirely. Slower per call (~2–3×) but
        reliable. Default False uses the dask-dispatched ``chunks={"y": -1,
        "x": -1}`` path.
    """
    if zarr_async_concurrency is not None:
        import zarr
        zarr.config.set({"async.concurrency": zarr_async_concurrency})

    if t is None and source_url is None:
        raise ValueError("Must provide one of t= or source_url=")

    # If only t is given, look up the source URL by listing the day's files and
    # finding the one whose *internal* t matches. (We can't rely on the
    # filename `_s_` token alone — it's the scan-command time, not mid-scan.)
    if source_url is None:
        source_url = _t_to_source_url(_ensure_datetime64_ns(t), registry=registry)

    # ---- fetch + open source — this gives us the canonical t ----
    # Asymmetric chunk settings:
    #
    #   - Source NetCDF on local disk: chunks=None (no dask). h5netcdf reads
    #     are fast local-file I/O and dask wrapping wouldn't help. The
    #     compare loop accesses .values lazily, one variable at a time.
    #
    #   - icechunk side: chunks={"y": -1, "x": -1} — one dask block per
    #     variable, NOT one block per underlying 226×226 zarr chunk. Each
    #     block-fetch task internally pulls all ~588 zarr chunks for that
    #     variable via icechunk's storage backend (whose async parallelism
    #     is controlled by zarr.async.concurrency). Dask scheduler sees ~32
    #     tasks per verify, not ~18K.
    #
    # .load() is called once on the t slice, kicking off all variables'
    # block-fetches as a single dask graph. With an active distributed
    # client, those block-fetches fan out to workers in parallel; the
    # driver gathers the materialised arrays back. Workers hold one
    # variable-block (~60 MB) at a time, the driver holds the gathered
    # arrays (~1.4 GB). The driver is the notebook process (deliberately
    # bigger than the small Coiled workers), so the memory cost lands
    # where there's headroom.
    #
    # The `with` blocks bound the underlying h5netcdf file handle and the
    # icechunk readonly session lifetime; .load() copies data into in-memory
    # numpy arrays so the lazy Datasets can be closed afterward.
    timings: dict[str, float] = {}

    def _time(name: str):
        @contextlib.contextmanager
        def _cm():
            t0 = time.perf_counter()
            yield
            timings[name] = time.perf_counter() - t0
        return _cm()

    with _time("fetch source"):
        tmp = _fetch_source_to_tempfile(source_url, registry=registry)
    try:
        with xr.open_dataset(tmp, engine="h5netcdf", chunks=None) as src_ds_lazy:
            file_t = _ensure_datetime64_ns(src_ds_lazy["t"].values)

            if t is not None and _ensure_datetime64_ns(t) != file_t:
                raise ValueError(
                    f"t={t!r} disagrees with the internal t of {source_url!r} (={file_t})."
                    f" (Hint: the filename `_s_` token is scan-command time, ~minutes "
                    f"earlier than the actual t.)"
                )
            t_ns = file_t

            # ---- open icechunk side ----
            # chunks=None (force_synchronous=True) bypasses dask entirely —
            # all chunk reads run on this process through icechunk's async
            # runtime. The default ``{"y": -1, "x": -1}`` wraps each variable
            # as one dask block so .load() can dispatch the block-fetches to
            # cluster workers in parallel — much faster but vulnerable to
            # stale credentials on workers.
            chunks_arg: dict | None = None if force_synchronous else {"y": -1, "x": -1}
            ro = repo.readonly_session(branch=branch)
            with xr.open_zarr(
                ro.store, group=group,
                chunks=chunks_arg, zarr_format=3,
            ) as ic_ds_lazy:
                if "t" not in ic_ds_lazy.dims:
                    raise ValueError(f"Group {group!r} has no 't' dim; cannot index by timestamp")
                if t_ns not in ic_ds_lazy["t"].values:
                    raise KeyError(
                        f"t={t_ns} (from source {source_url.rsplit('/', 1)[-1]}) not present "
                        f"in icechunk group {group!r} on branch {branch!r}. "
                        f"Available range: {ic_ds_lazy['t'].values[0]} → {ic_ds_lazy['t'].values[-1]}, "
                        f"n_t={ic_ds_lazy.sizes['t']}"
                    )
                ic_slice_lazy = ic_ds_lazy.sel(t=t_ns)

                # Materialise the icechunk side on the driver via the
                # active dask client. Each variable is a single dask block
                # (chunks={"y": -1, "x": -1}), so the cluster sees ~32 tasks,
                # not the ~18K we'd get with default per-zarr-chunk blocking.
                with _time("load icechunk"):
                    ic_slice = ic_slice_lazy.load()

                with _time("compare"):
                    report = _run_comparison(
                        ic_slice=ic_slice, src_ds=src_ds_lazy, t=t_ns, source_url=source_url,
                        group=group, variables=variables, skip_variables=skip_variables,
                        verbose=False,  # caller handles the success print below
                    )

                if verbose:
                    print(
                        f"✓ verify_timestep t={t_ns}  group={group!r}\n"
                        f"  source: {source_url.rsplit('/', 1)[-1]}\n"
                        f"  compared {len(report.variables_compared)} vars; "
                        f"only-in-icechunk={len(report.variables_only_in_icechunk)}, "
                        f"only-in-source={len(report.variables_only_in_source)}\n"
                        f"  timings: "
                        + ", ".join(f"{k}={v:.1f}s" for k, v in timings.items())
                    )
                return report
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass


# Source-side per-channel coord families that the pipeline consolidates into
# a single icechunk-side coord (see consolidate_band_coords in mcmipf_pipeline).
# Each tuple: (icechunk_coord_name, source_prefix, rtol_for_compare, atol_for_compare).
# - band: exact integer match, no tolerance needed.
# - wavelength: NOAA changed the precision of the band_wavelength_C{NN} attrs
#   between 2017 and 2023 (early-era files report e.g. 13.27 μm for C16, later
#   files use the published nominal 13.3 μm). The stored icechunk value comes
#   from a 2023 reference file and matches the GOES-R ABI published nominal
#   central wavelengths — but pre-op-era files have slightly less-precise
#   values. Allow ~1% relative + ~0.1 μm absolute tolerance so we still catch
#   any genuine wavelength corruption but accept NOAA's metadata drift. See
#   GOES_DATA_INCONSISTENCIES.md.
_CONSOLIDATED_COORDS: tuple[tuple[str, str, float, float], ...] = (
    ("band",       "band_id_C",         0.0,   0.0),
    ("wavelength", "band_wavelength_C", 0.01,  0.1),
)


def _compare_consolidated_band_coords(
    ic_slice: xr.Dataset, src_ds: xr.Dataset,
) -> tuple[list[FieldMismatch], list[str]]:
    """Cross-check each source per-channel band_id_C{NN} / band_wavelength_C{NN}
    against the corresponding element of the icechunk consolidated array.

    Returns ``(mismatches, handled_source_names)``. ``handled_source_names``
    should be subtracted from ``only_src`` so we don't double-flag those names
    as "missing in icechunk".
    """
    mismatches: list[FieldMismatch] = []
    handled: list[str] = []

    for ic_name, src_prefix, rtol, atol in _CONSOLIDATED_COORDS:
        if ic_name not in ic_slice.variables:
            continue
        ic_arr = np.asarray(ic_slice[ic_name].values)
        for i in range(1, 17):
            src_name = f"{src_prefix}{i:02d}"
            if src_name not in src_ds.variables:
                continue
            handled.append(src_name)
            src_val = src_ds[src_name].values.flat[0]
            if (i - 1) >= ic_arr.size:
                mismatches.append(FieldMismatch(
                    src_name, "values",
                    f"icechunk {ic_name} has only {ic_arr.size} elements; "
                    f"can't compare to band {i}",
                ))
                continue
            ic_val = ic_arr[i - 1]
            # Use coord-specific tolerance: 0/0 for band_id (exact), looser
            # for wavelength (NOAA metadata drift).
            if rtol == 0.0 and atol == 0.0:
                ok = _scalar_or_array_equal(ic_val, src_val)
            else:
                try:
                    ok = bool(np.allclose(ic_val, src_val, rtol=rtol, atol=atol,
                                          equal_nan=True))
                except TypeError:
                    ok = bool(np.allclose(ic_val, src_val, rtol=rtol, atol=atol))
            if not ok:
                mismatches.append(FieldMismatch(
                    src_name, "values",
                    f"icechunk {ic_name}[{i-1}] = {ic_val!r}, source {src_name} = {src_val!r}",
                ))

    return mismatches, handled


def _run_comparison(
    *,
    ic_slice: xr.Dataset,
    src_ds: xr.Dataset,
    t: np.datetime64,
    source_url: str,
    group: str,
    variables: Sequence[str] | None,
    skip_variables: Sequence[str],
    verbose: bool,
) -> VerificationReport:
    ic_vars = set(ic_slice.data_vars) | set(ic_slice.coords)
    src_vars = set(src_ds.data_vars) | set(src_ds.coords)

    overlap = ic_vars & src_vars
    only_ic = sorted(ic_vars - src_vars)
    only_src = sorted(src_vars - ic_vars)

    if variables is not None:
        overlap &= set(variables)

    overlap -= set(skip_variables)

    mismatches: list[FieldMismatch] = []
    compared: list[str] = []
    for name in sorted(overlap):
        try:
            mismatches.extend(_compare_variable(name, ic_slice[name], src_ds[name]))
            compared.append(name)
        except Exception as e:
            mismatches.append(FieldMismatch(name, "missing", f"compare failed: {e!r}"))

    # Handle the consolidated band/wavelength coords specially before flagging
    # any source-only names as missing.
    consolidated_mismatches, handled = _compare_consolidated_band_coords(ic_slice, src_ds)
    mismatches.extend(consolidated_mismatches)
    compared.extend(handled)
    only_src = sorted(set(only_src) - set(handled))

    # Variables only in the source but absent from icechunk are real failures —
    # we didn't ingest something. Variables only in icechunk (e.g. synthesized
    # fill channels for files that were missing them) are expected, so they
    # only get surfaced as a count in the report.
    for name in only_src:
        if variables is not None and name not in variables:
            continue
        if name in skip_variables:
            continue
        mismatches.append(FieldMismatch(
            name, "missing",
            "present in source NetCDF, absent from icechunk group",
        ))

    report = VerificationReport(
        t=t,
        source_url=source_url,
        group=group,
        variables_compared=compared,
        variables_only_in_icechunk=only_ic,
        variables_only_in_source=only_src,
    )

    if mismatches:
        raise VerificationError(t=t, source_url=source_url, group=group, mismatches=mismatches)

    if verbose:
        print(
            f"✓ verify_timestep t={t}  group={group!r}\n"
            f"  source: {source_url.rsplit('/', 1)[-1]}\n"
            f"  compared {len(compared)} vars; "
            f"only-in-icechunk={len(only_ic)}, only-in-source={len(only_src)}"
        )
    return report


# =============================================================================
# CLI
# =============================================================================

def _cli() -> None:
    import argparse
    p = argparse.ArgumentParser(
        description=(
            "Compare a single committed scan in an icechunk group against its "
            "source NetCDF on S3. Exits 0 on agreement, 1 on any mismatch."
        ),
        epilog=(
            "Examples:\n"
            "  uv run mcmipf_verifier/mcmipf_verifier.py \\\n"
            "      --group ABI-L2-MCMIPF/post-2023-04-19 \\\n"
            "      --t 2023-04-19T15:00:20.500\n"
            "  uv run mcmipf_verifier/mcmipf_verifier.py \\\n"
            "      --group ABI-L2-MCMIPF/post-2023-04-19 \\\n"
            "      --source-url s3://noaa-goes16/ABI-L2-MCMIPF/2023/109/15/OR_..._s20231091500205_..._c20231091510007.nc\n"
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
    )
    p.add_argument("--repo", default="earthmover-public/goes-16",
                   help="Arraylake repo (default: earthmover-public/goes-16)")
    p.add_argument("--branch", default="main")
    p.add_argument("--group", required=True,
                   help="Zarr group within the repo, e.g. ABI-L2-MCMIPF/post-2023-04-19")
    src = p.add_mutually_exclusive_group(required=True)
    src.add_argument("--t", help="Scan-start timestamp, parseable by np.datetime64")
    src.add_argument("--source-url", help="Full s3:// URL of the source NetCDF")
    p.add_argument("--variables", nargs="+", help="Restrict comparison to these vars")
    p.add_argument("--skip", nargs="+", default=[], help="Skip these vars")
    p.add_argument("--max-report", type=int, default=50,
                   help="Maximum number of mismatches to print on failure (default 50)")
    args = p.parse_args()

    from arraylake import Client
    repo = Client().get_repo(args.repo)

    try:
        verify_timestep(
            repo=repo,
            branch=args.branch,
            group=args.group,
            t=args.t if args.t else None,
            source_url=args.source_url,
            variables=args.variables,
            skip_variables=args.skip,
        )
    except VerificationError as e:
        # Pretty-print, truncating very long mismatch lists.
        n = len(e.mismatches)
        shown = e.mismatches[: args.max_report]
        print(f"\n✗ VerificationError: {n} mismatch(es) at t={e.t} in {e.group}")
        print(f"  source: {e.source_url}")
        for m in shown:
            print(str(m))
        if n > args.max_report:
            print(f"  ... ({n - args.max_report} more mismatches suppressed; "
                  f"raise --max-report to see them)")
        sys.exit(1)

## Schema validation and preprocessing

In [14]:
"""End-to-end preprocessing pipeline for GOES-16 ABI-L2-MCMIPF virtual datasets.

Self-contained — copy this whole module (or paste its contents into a notebook
cell) and call :func:`preprocess` per file, or pass it directly to
:func:`virtualizarr.open_virtual_mfdataset`:

    vmds = vz.open_virtual_mfdataset(
        urls,
        registry=registry,
        parser=vz.parsers.HDFParser(),
        loadable_variables=list(DEFAULT_LOADABLE_VARIABLES),
        preprocess=preprocess,
        concat_dim="t", combine="nested",
    )

The pipeline is a series of chainable ``xr.Dataset -> xr.Dataset`` functions:

    validate_raw                — pandera structural validation + codec
                                  validation of the raw virtualizarr-opened
                                  file. Pass-through.
    fill_missing_channels       — back-fill any missing per-channel variable.
    fill_missing_scalar_metadata — synthesize NaN/0/NaT placeholders for any
                                  expected scalar telemetry variable absent
                                  from the source.
    finalize_per_channel        — per-channel encoding fixups (restore int32 on
                              ``outlier_pixel_count_C{NN}``, etc.). Every
                              per-channel data variable stays as its own
                              xarray variable — no band-dim stacking.
    consolidate_band_coords  — replace the 16 trivial ``band_id_C{NN}`` and
                              16 ``band_wavelength_C{NN}`` coords with one
                              ``band(band)`` and one ``wavelength(band)``
                              coord of size 16.
    add_time_dimension       — ``expand_dims('t')`` on every data variable using
                              the existing scalar ``t`` coord.
    validate_cleaned         — pandera structural validation of the cleaned dataset.

:func:`preprocess` chains them all and raises immediately on the first failure.
pandera's ``lazy=True`` still collects every schema-level error within a single
validation call, so the report from whichever stage fails first shows every
issue it found. The raised :class:`MCMIPFValidationError` attaches the source
filename (from ``ds.encoding['source']`` or ``ds.attrs['dataset_name']``).
"""

from __future__ import annotations

import datetime
from collections.abc import Iterable
from typing import Any

import numpy as np
import pandera.xarray as pa
import xarray as xr
from virtualizarr.manifests import ChunkManifest, ManifestArray
from zarr.codecs import BytesCodec, Shuffle, Zlib
from zarr.core.chunk_key_encodings import DefaultChunkKeyEncoding
from zarr.core.dtype import Int8, Int16
from zarr.core.metadata.v3 import ArrayV3Metadata


# zarr renamed RegularChunkGrid -> RegularChunkGridMetadata (and moved it from
# zarr.core.chunk_grids to zarr.core.metadata.v3) between 3.1.x and 3.2.x.
try:  # zarr >= 3.2
    from zarr.core.metadata.v3 import RegularChunkGridMetadata as _ChunkGrid
except ImportError:  # zarr <= 3.1
    from zarr.core.chunk_grids import RegularChunkGrid as _ChunkGrid


# =============================================================================
# Constants
# =============================================================================
X_SIZE = 5424
Y_SIZE = 5424
N_BANDS = 16
ALL_CHANNELS = range(1, 17)
REFLECTIVE_CHANNELS = range(1, 7)   # C01-C06 — reflectance_factor stats
EMISSIVE_CHANNELS = range(7, 17)    # C07-C16 — brightness_temperature stats

DATETIME_NS = np.dtype("datetime64[ns]")


def _ch(i: int) -> str:
    return f"C{i:02d}"


# Recommended loadable_variables — every low-dimensional variable.
DEFAULT_LOADABLE_VARIABLES: tuple[str, ...] = (
    "x", "y", "t", "band",
    "x_image", "y_image",
    "x_image_bounds", "y_image_bounds", "time_bounds",
    "goes_imager_projection",
    "nominal_satellite_height",
    "nominal_satellite_subpoint_lat",
    "nominal_satellite_subpoint_lon",
    "geospatial_lat_lon_extent",
    "percent_uncorrectable_GRB_errors",
    "percent_uncorrectable_L0_errors",
    "algorithm_product_version_container",
    "dynamic_algorithm_input_data_container",
    *(f"band_id_{_ch(i)}"             for i in ALL_CHANNELS),
    *(f"band_wavelength_{_ch(i)}"     for i in ALL_CHANNELS),
    *(f"outlier_pixel_count_{_ch(i)}" for i in ALL_CHANNELS),
    *(f"{s}_reflectance_factor_{_ch(i)}"
      for s in ("min", "max", "mean", "std_dev")
      for i in REFLECTIVE_CHANNELS),
    *(f"{s}_brightness_temperature_{_ch(i)}"
      for s in ("min", "max", "mean", "std_dev")
      for i in EMISSIVE_CHANNELS),
)

# Per-channel name prefixes (the suffix is supplied by `_ch(i)`, e.g. "C01").
_PER_CHANNEL_2D = ("CMI_", "DQF_")
_PER_CHANNEL_BAND_LOADED = ("band_id_", "band_wavelength_")
_PER_CHANNEL_SCALAR_LOADED = (
    "outlier_pixel_count_",
    "min_reflectance_factor_", "max_reflectance_factor_",
    "mean_reflectance_factor_", "std_dev_reflectance_factor_",
    "min_brightness_temperature_", "max_brightness_temperature_",
    "mean_brightness_temperature_", "std_dev_brightness_temperature_",
)

# Non-per-channel scalar / 1-D metadata variables we expect in every source
# file. Used by both schemas and by fill_missing_scalar_metadata, which
# synthesizes NaN/0 placeholders for any that are absent (without this, an
# xr.concat across a batch where some files miss a var would fail).
_EXPECTED_SHARED_VARS: tuple[tuple[str, Any, tuple[str, ...]], ...] = (
    ("algorithm_product_version_container",    np.int32,    ()),
    ("dynamic_algorithm_input_data_container", np.int32,    ()),
    ("geospatial_lat_lon_extent",              np.float32,  ()),
    ("goes_imager_projection",                 np.int32,    ()),
    ("nominal_satellite_height",               np.float32,  ()),
    ("nominal_satellite_subpoint_lat",         np.float32,  ()),
    ("nominal_satellite_subpoint_lon",         np.float32,  ()),
    ("percent_uncorrectable_GRB_errors",       np.float32,  ()),
    ("percent_uncorrectable_L0_errors",        np.float32,  ()),
    ("time_bounds",    DATETIME_NS, ("number_of_time_bounds",)),
    ("x_image_bounds", np.float32,  ("number_of_image_bounds",)),
    ("y_image_bounds", np.float32,  ("number_of_image_bounds",)),
)

# Sizes for the named dims used in _EXPECTED_SHARED_VARS — used when
# synthesizing placeholders for fully-missing variables (the dim might not
# already exist in the source's ds.dims).
_SHARED_DIM_SIZES: dict[str, int] = {
    "number_of_time_bounds": 2,
    "number_of_image_bounds": 2,
}

# Canonical CF attrs that NOAA's earlier-era source files lacked (later files
# added them). finalize_per_channel applies these as defaults via setdefault
# so any value already present in source is preserved. See
# GOES_DATA_INCONSISTENCIES.md for the observed drift.
_CANONICAL_VAR_ATTRS: dict[str, dict[str, Any]] = {
    "x_image_bounds": {"units": "rad"},
    "y_image_bounds": {"units": "rad"},
}

# Canonical CF attrs that NOAA *monotonically expanded* over time — older
# source files have narrower values, newer ones have the wider/canonical
# value. finalize_per_channel writes these unconditionally (override mode),
# so the icechunk-stored attr is always the modern wider value regardless of
# which file contributed first. The verifier then accepts source-side
# narrower values as a subset of icechunk's range.
#
# Currently: DQF_C{NN}.valid_range expanded from [0, 3] to [0, 4] when NOAA
# added a new DQF flag value (4) — see GOES_DATA_INCONSISTENCIES.md.
_CANONICAL_VAR_ATTR_OVERRIDES: dict[str, dict[str, Any]] = {
    f"DQF_C{i:02d}": {"valid_range": np.array([0, 4], dtype=np.int8)}
    for i in ALL_CHANNELS
}

# Expected fill_value for each per-channel 2-D variable. These come from the
# GOES MCMIPF product definition and match the per-chunk fill_value embedded in
# the source HDF5 files.
EXPECTED_FILL_VALUES: dict[str, Any] = {
    "CMI_": np.int16(-1),
    "DQF_": np.int8(-1),
}


# ---------------------------------------------------------------------------
# Hard-coded zarr metadata for synthesizing fill-only ManifestArrays from
# scratch, used by fill_missing_channels when no per-prefix variable exists
# in the file to serve as a template. Values match the GOES MCMIPF product
# definition (and the codec/fill_value tables above).
# ---------------------------------------------------------------------------
_MCMIPF_2D_CHUNKS = (226, 226)
_MCMIPF_2D_GRID = (Y_SIZE // _MCMIPF_2D_CHUNKS[0], X_SIZE // _MCMIPF_2D_CHUNKS[1])


def _make_2d_metadata(
    *,
    data_type: Any,
    fill_value: Any,
    codecs: tuple,
) -> ArrayV3Metadata:
    return ArrayV3Metadata(
        shape=(Y_SIZE, X_SIZE),
        data_type=data_type,
        chunk_grid=_ChunkGrid(chunk_shape=_MCMIPF_2D_CHUNKS),
        chunk_key_encoding=DefaultChunkKeyEncoding(separator="/"),
        fill_value=fill_value,
        codecs=codecs,
        attributes={},
        dimension_names=None,
        storage_transformers=(),
    )


# Post-era (Shuffle present)
CMI_METADATA_POST: ArrayV3Metadata = _make_2d_metadata(
    data_type=Int16(endianness="little"),
    fill_value=np.int16(-1),
    codecs=(BytesCodec(endian="little"), Shuffle(elementsize=2), Zlib(level=1)),
)

DQF_METADATA_POST: ArrayV3Metadata = _make_2d_metadata(
    data_type=Int8(),
    fill_value=np.int8(-1),
    codecs=(BytesCodec(endian=None), Shuffle(elementsize=1), Zlib(level=1)),
)

# Pre-era (no Shuffle filter)
CMI_METADATA_PRE: ArrayV3Metadata = _make_2d_metadata(
    data_type=Int16(endianness="little"),
    fill_value=np.int16(-1),
    codecs=(BytesCodec(endian="little"), Zlib(level=1)),
)

DQF_METADATA_PRE: ArrayV3Metadata = _make_2d_metadata(
    data_type=Int8(),
    fill_value=np.int8(-1),
    codecs=(BytesCodec(endian=None), Zlib(level=1)),
)

_PER_CHANNEL_2D_METADATA_POST: dict[str, ArrayV3Metadata] = {
    "CMI_": CMI_METADATA_POST,
    "DQF_": DQF_METADATA_POST,
}
_PER_CHANNEL_2D_METADATA_PRE: dict[str, ArrayV3Metadata] = {
    "CMI_": CMI_METADATA_PRE,
    "DQF_": DQF_METADATA_PRE,
}

# Back-compat aliases for code that imported the old single-era names.
CMI_METADATA = CMI_METADATA_POST
DQF_METADATA = DQF_METADATA_POST
_PER_CHANNEL_2D_METADATA = _PER_CHANNEL_2D_METADATA_POST


def _make_fill_manifest_array(metadata: ArrayV3Metadata) -> ManifestArray:
    """Construct a fill-only ``ManifestArray`` from a canonical metadata.

    Every chunk in the result has ``path=""`` so reads return ``fill_value``
    (per the Zarr V3 missing-chunk semantics). Used to synthesize placeholders
    for channels that are entirely absent from the source file.
    """
    paths = np.full(_MCMIPF_2D_GRID, "", dtype=np.dtypes.StringDType())
    offsets = np.zeros(_MCMIPF_2D_GRID, dtype=np.uint64)
    lengths = np.zeros(_MCMIPF_2D_GRID, dtype=np.uint64)
    manifest = ChunkManifest.from_arrays(paths=paths, offsets=offsets, lengths=lengths)
    return ManifestArray(metadata=metadata, chunkmanifest=manifest)


# Expected codec pipelines on the virtual variables (zarr-v3 codecs from HDFParser).
# Post-era (≥ 2023-04-19 15:00 UTC) uses a 3-codec pipeline with Shuffle;
# pre-era uses a 2-codec pipeline without Shuffle. Selected per-file by
# :func:`_era_for_ds`.
_EXPECTED_CODECS_BY_PREFIX_POST: dict[str, list[dict[str, Any]]] = {
    "CMI_C": [
        {"class": "BytesCodec", "endian": "little"},
        {"class": "Shuffle", "codec_name": "numcodecs.shuffle",
         "codec_config": {"elementsize": 2}},
        {"class": "Zlib",    "codec_name": "numcodecs.zlib",
         "codec_config": {"level": 1}},
    ],
    "DQF_C": [
        {"class": "BytesCodec", "endian": None},
        {"class": "Shuffle", "codec_name": "numcodecs.shuffle",
         "codec_config": {"elementsize": 1}},
        {"class": "Zlib",    "codec_name": "numcodecs.zlib",
         "codec_config": {"level": 1}},
    ],
}

_EXPECTED_CODECS_BY_PREFIX_PRE: dict[str, list[dict[str, Any]]] = {
    "CMI_C": [
        {"class": "BytesCodec", "endian": "little"},
        {"class": "Zlib",    "codec_name": "numcodecs.zlib",
         "codec_config": {"level": 1}},
    ],
    "DQF_C": [
        {"class": "BytesCodec", "endian": None},
        {"class": "Zlib",    "codec_name": "numcodecs.zlib",
         "codec_config": {"level": 1}},
    ],
}

# Back-compat alias: external callers that imported the old name continue to
# get the post-era spec.
_EXPECTED_CODECS_BY_PREFIX = _EXPECTED_CODECS_BY_PREFIX_POST


# =============================================================================
# Custom error
# =============================================================================
class MCMIPFValidationError(Exception):
    """Pipeline failure annotated with the source filename.

    Wraps the underlying exception (the original is accessible via ``__cause__``
    and the ``original`` attribute). pandera's ``lazy=True`` still collects every
    error within a single validation call into one ``SchemaErrors`` instance, so
    you'll see all schema issues from whichever stage failed first.
    """

    def __init__(self, source: str, original: Exception):
        self.source = source
        self.original = original
        super().__init__(f"\nPipeline failed for {source}:\n\n{original}")


def _source_of(ds: xr.Dataset) -> str:
    # open_mfdataset stamps encoding["source"]; standalone open_virtual_dataset
    # doesn't — but the GOES file itself carries the filename in attrs.
    return (
        ds.encoding.get("source")
        or ds.attrs.get("dataset_name")
        or "<unknown>"
    )


def _era_for_ds(ds: xr.Dataset) -> str:
    """Return ``"pre"`` or ``"post"`` based on whether the file's filename
    scan-start time is before or after :data:`CODEC_CHANGE_DATETIME`.

    Tries the URL basename of ``ds.encoding["source"]`` first, then
    ``ds.attrs["dataset_name"]``. Raises :class:`ValueError` if neither
    parses — silently defaulting could cause a codec mismatch to be missed.
    """
    for candidate in (
        (ds.encoding.get("source") or "").rsplit("/", 1)[-1],
        ds.attrs.get("dataset_name") or "",
    ):
        if not candidate:
            continue
        try:
            scan_start = _parse_scan_start_to_datetime(candidate)
        except (ValueError, IndexError):
            continue
        return "post" if scan_start >= CODEC_CHANGE_DATETIME else "pre"
    raise ValueError(
        "Cannot determine codec era: neither ds.encoding['source'] nor "
        "ds.attrs['dataset_name'] yields a parseable scan-start token."
    )


# =============================================================================
# Raw-file schema
# =============================================================================
def make_mcmipf_schema(
    loadable_variables: Iterable[str] = DEFAULT_LOADABLE_VARIABLES,
) -> pa.DatasetSchema:
    """Pandera schema for a virtualizarr-opened MCMIPF file (pre-cleaning)."""
    loadable = frozenset(loadable_variables)

    def array_type_for(name: str) -> Any:
        return np.ndarray if name in loadable else ManifestArray

    per_channel: dict[str, pa.DataVar] = {}
    for i in ALL_CHANNELS:
        c = _ch(i)
        per_channel[f"CMI_{c}"] = pa.DataVar(
            dtype=np.int16, dims=("y", "x"),
            array_type=array_type_for(f"CMI_{c}"), required=False,
        )
        per_channel[f"DQF_{c}"] = pa.DataVar(
            dtype=np.int8, dims=("y", "x"),
            array_type=array_type_for(f"DQF_{c}"), required=False,
        )
        # outlier_pixel_count and the per-channel stats can be NaN in real
        # source files when a channel had no valid pixels in that scan; mark
        # them nullable so a single bad-data scan doesn't fail validate_raw.
        per_channel[f"outlier_pixel_count_{c}"] = pa.DataVar(
            dtype=np.float64, dims=(),
            array_type=array_type_for(f"outlier_pixel_count_{c}"),
            required=False, nullable=True,
        )
        per_channel[f"band_id_{c}"] = pa.DataVar(
            dtype=np.int8, dims=("band",),
            array_type=array_type_for(f"band_id_{c}"), required=False,
        )
        per_channel[f"band_wavelength_{c}"] = pa.DataVar(
            dtype=np.float32, dims=("band",),
            array_type=array_type_for(f"band_wavelength_{c}"), required=False,
        )
    for stat in ("min", "max", "mean", "std_dev"):
        for i in REFLECTIVE_CHANNELS:
            name = f"{stat}_reflectance_factor_{_ch(i)}"
            per_channel[name] = pa.DataVar(
                dtype=np.float32, dims=(),
                array_type=array_type_for(name),
                required=False, nullable=True,
            )
        for i in EMISSIVE_CHANNELS:
            name = f"{stat}_brightness_temperature_{_ch(i)}"
            per_channel[name] = pa.DataVar(
                dtype=np.float32, dims=(),
                array_type=array_type_for(name),
                required=False, nullable=True,
            )

    # Single source of truth — see _EXPECTED_SHARED_VARS at the top of the
    # module. fill_missing_scalar_metadata also iterates the same tuple.
    shared_spec = list(_EXPECTED_SHARED_VARS)
    # Mark every scalar / 1-D metadata variable both nullable and optional.
    # Real-world NOAA files occasionally ship one or more of these telemetry
    # scalars as NaN (observed in early-2017 files for
    # percent_uncorrectable_GRB_errors), and any given file could in
    # principle be missing one entirely. Rather than tracking which specific
    # ones can go null/absent, we accept that any of them might and rely on
    # the calibration / codec / value verifiers to catch real defects. See
    # GOES_DATA_INCONSISTENCIES.md.
    shared = {
        name: pa.DataVar(
            dtype=dtype, dims=dims, array_type=array_type_for(name),
            nullable=True, required=False,
        )
        for name, dtype, dims in shared_spec
    }

    coords = {
        "x":       pa.Coordinate(dtype=np.float64, dimension=True,  indexed=True),
        "y":       pa.Coordinate(dtype=np.float64, dimension=True,  indexed=True),
        "t":       pa.Coordinate(dtype=DATETIME_NS, dimension=False, indexed=False),
        "x_image": pa.Coordinate(dtype=np.float32, dimension=False, indexed=False),
        "y_image": pa.Coordinate(dtype=np.float32, dimension=False, indexed=False),
    }

    return pa.DatasetSchema(
        data_vars={**shared, **per_channel},
        coords=coords,
        sizes={
            "x": X_SIZE,
            "y": Y_SIZE,
            "band": 1,
            "number_of_time_bounds": 2,
            "number_of_image_bounds": 2,
        },
        strict=True,
        strict_coords=False,
    )


# =============================================================================
# Codec validation (raw)
# =============================================================================
def _codec_matches(codec: Any, expected: dict[str, Any]) -> bool:
    if type(codec).__name__ != expected["class"]:
        return False
    for key, want in expected.items():
        if key == "class":
            continue
        got = getattr(codec, key, None)
        if hasattr(got, "value"):           # enum (e.g. Endian.little)
            got = got.value
        if key == "codec_config":
            got_d = dict(got) if got is not None else {}
            for k, v in want.items():
                if got_d.get(k) != v:
                    return False
            continue
        if got != want:
            return False
    return True


def _expected_codecs_for(var_name: str, era: str) -> list[dict[str, Any]] | None:
    table = (
        _EXPECTED_CODECS_BY_PREFIX_POST if era == "post"
        else _EXPECTED_CODECS_BY_PREFIX_PRE
    )
    for prefix, expected in table.items():
        if var_name.startswith(prefix):
            return expected
    return None


def _check_codecs(ds: xr.Dataset) -> None:
    """Raise ``ValueError`` listing every codec mismatch / unexpected virtual var.

    Selects the expected codec pipeline based on the file's codec era
    (pre vs post 2023-04-19 boundary) inferred from its filename.
    """
    era = _era_for_ds(ds)
    errors: list[str] = []
    for name, da in {**ds.data_vars, **ds.coords}.items():
        if not isinstance(da.data, ManifestArray):
            continue
        expected = _expected_codecs_for(name, era=era)
        if expected is None:
            errors.append(
                f"{name}: unexpectedly virtual — only CMI_C* and DQF_C* are "
                f"supposed to remain as ManifestArray"
            )
            continue
        actual = list(da.data.metadata.codecs)
        if len(actual) != len(expected):
            errors.append(
                f"{name}: expected {len(expected)} codecs, got {len(actual)}: {actual!r}"
            )
            continue
        for i, (act, exp) in enumerate(zip(actual, expected)):
            if not _codec_matches(act, exp):
                errors.append(
                    f"{name}: codec[{i}] mismatch — expected {exp}, got {act!r}"
                )
    if errors:
        raise ValueError("Codec validation failed:\n  " + "\n  ".join(errors))


# =============================================================================
# Cleaned-dataset schema
# =============================================================================
def make_cleaned_mcmipf_schema() -> pa.DatasetSchema:
    """Pandera schema for the dataset produced by :func:`preprocess`.

    Every per-channel variable in the source NetCDF stays as its own xarray
    variable — no band-dim stacking. This preserves the source's CF attrs
    (long_name, valid_range, etc.) verbatim per channel.
    """
    # 2-D imagery per channel — (t, y, x) ManifestArrays.
    cmi_vars = {
        f"CMI_{_ch(i)}": pa.DataVar(
            dtype=np.int16, dims=("t", "y", "x"), array_type=ManifestArray,
        )
        for i in ALL_CHANNELS
    }
    dqf_vars = {
        f"DQF_{_ch(i)}": pa.DataVar(
            dtype=np.int8, dims=("t", "y", "x"), array_type=ManifestArray,
        )
        for i in ALL_CHANNELS
    }

    # Per-channel loaded scalars. outlier_pixel_count picks up NaN when a
    # synthesized fill stands in for an absent source channel.
    outlier_vars = {
        f"outlier_pixel_count_{_ch(i)}": pa.DataVar(
            dtype=np.float64, dims=("t",), array_type=np.ndarray, nullable=True,
        )
        for i in ALL_CHANNELS
    }

    # Per-channel stats. The source NetCDF publishes all 4 stats for every
    # channel (with NaN on the inapplicable ones — brightness stats on the
    # reflective bands C01-C06, reflectance stats on the emissive bands
    # C07-C16). DEFAULT_LOADABLE_VARIABLES only loads the *applicable* ones,
    # but fill_missing_channels synthesizes the rest as NaN via xr.full_like,
    # so the cleaned dataset carries all 16 channels of both.
    refl_stat_vars = {
        f"{stat}_reflectance_factor_{_ch(i)}": pa.DataVar(
            dtype=np.float32, dims=("t",), array_type=np.ndarray, nullable=True,
        )
        for stat in ("min", "max", "mean", "std_dev")
        for i in ALL_CHANNELS
    }
    bt_stat_vars = {
        f"{stat}_brightness_temperature_{_ch(i)}": pa.DataVar(
            dtype=np.float32, dims=("t",), array_type=np.ndarray, nullable=True,
        )
        for stat in ("min", "max", "mean", "std_dev")
        for i in ALL_CHANNELS
    }

    # Consolidated band axis. Replaces the 16 per-channel band_id_C{NN} +
    # 16 band_wavelength_C{NN} that the source NetCDF carries — those become
    # the band dim coord (values 1..16) and the wavelength(band) non-dim
    # coord respectively, materialised in consolidate_band_coords.

    # Scalar / 1-D telemetry vars — single source of truth at module top
    # (_EXPECTED_SHARED_VARS). All are nullable + optional, see the comment
    # in make_mcmipf_schema for rationale (real-world NOAA files ship some
    # as NaN or missing entirely; fill_missing_scalar_metadata synthesizes
    # placeholders for missing ones before this validation).
    shared_vars = {
        name: pa.DataVar(
            dtype=dtype, dims=("t",) + dims, array_type=np.ndarray,
            nullable=True, required=False,
        )
        for name, dtype, dims in _EXPECTED_SHARED_VARS
    }

    return pa.DatasetSchema(
        data_vars={
            **cmi_vars,
            **dqf_vars,
            **outlier_vars,
            **refl_stat_vars,
            **bt_stat_vars,
            **shared_vars,
        },
        coords={
            "t":          pa.Coordinate(dtype=DATETIME_NS, dimension=True,  indexed=True),
            "y":          pa.Coordinate(dtype=np.float64, dimension=True,  indexed=True),
            "x":          pa.Coordinate(dtype=np.float64, dimension=True,  indexed=True),
            "band":       pa.Coordinate(dtype=np.int8,    dimension=True,  indexed=True),
            "wavelength": pa.Coordinate(dtype=np.float32, dimension=False, indexed=False),
            "x_image":    pa.Coordinate(dtype=np.float32, dimension=False, indexed=False),
            "y_image":    pa.Coordinate(dtype=np.float32, dimension=False, indexed=False),
        },
        sizes={
            "t": 1,
            "band": N_BANDS,
            "y": Y_SIZE,
            "x": X_SIZE,
            "number_of_time_bounds": 2,
            "number_of_image_bounds": 2,
        },
        strict=True,
        strict_coords=False,
    )


# =============================================================================
# Chainable cleaning helpers
# =============================================================================
def _first_present(ds: xr.Dataset, prefix: str) -> str | None:
    for i in ALL_CHANNELS:
        name = f"{prefix}{_ch(i)}"
        if name in ds.variables:
            return name
    return None


def _fill_for_dtype(dtype: np.dtype) -> Any:
    if np.issubdtype(dtype, np.floating):
        return np.nan
    return 0


# =============================================================================
# Chainable pipeline functions  (Dataset -> Dataset)
# =============================================================================
def _parse_scan_start_to_datetime(source: str) -> np.datetime64:
    """Parse the ``sYYYYDDDHHMMSST`` filename token into ``datetime64[ns]``."""
    fname = source.rsplit("/", 1)[-1]
    token = fname.split("_s")[1].split("_")[0]
    if len(token) != 14:
        raise ValueError(f"Unexpected scan-start token length: {token!r}")
    year = int(token[0:4])
    doy = int(token[4:7])
    hour = int(token[7:9])
    minute = int(token[9:11])
    sec = int(token[11:13])
    tenths = int(token[13:14])
    date = datetime.date(year, 1, 1) + datetime.timedelta(days=doy - 1)
    return np.datetime64(
        datetime.datetime(date.year, date.month, date.day, hour, minute, sec, tenths * 100_000),
        "ns",
    )


# Maximum allowed delta between the filename's scan-start token and the file's
# internal `t` value. A real M6 scan has `t` at the scan midpoint, so the
# expected delta is ~5 minutes; we use 1 hour as a generous sanity envelope.
_MAX_T_VS_FILENAME_DRIFT = np.timedelta64(1, "h")


def validate_raw(
    ds: xr.Dataset,
    *,
    loadable_variables: Iterable[str] = DEFAULT_LOADABLE_VARIABLES,
) -> xr.Dataset:
    """Validate the raw file structure + codecs + calibration + t-value sanity.
    Returns ``ds`` unchanged.

    Raises immediately on the first failing check; pandera's ``lazy=True`` ensures
    all schema-level errors are still gathered in the one ``SchemaErrors`` exception.
    """
    make_mcmipf_schema(loadable_variables).validate(ds, lazy=True)
    _check_codecs(ds)
    # Verify the per-channel CMI calibration attrs match the expected era,
    # picked from the file's scalar t. Catches any unknown sub-era we
    # haven't characterised yet, including future NOAA-shipped drift.
    check_calibration(ds)

    # t-value sanity: catches sentinel/J2000 timestamps left by aborted scans
    # that snuck past the filename-level filter in mcmipf_archive.
    t = ds["t"].values
    epoch_threshold = np.datetime64("2017-01-01", "ns")
    if t < epoch_threshold:
        raise ValueError(
            f"`t` coordinate is {t!s} (before {epoch_threshold!s}) — likely an "
            f"aborted/placeholder scan with a sentinel timestamp."
        )

    # Cross-check the file's internal `t` against the scan-start time embedded
    # in its filename. Catches files whose internal `t` was mis-stamped
    # (drifted from the file's own filename) — a hole the sentinel check above
    # cannot detect. Try the source URL's basename first, then the
    # `dataset_name` attribute (locally-renamed files won't have the token in
    # the URL basename but do still have it in the attribute).
    scan_start: np.datetime64 | None = None
    for candidate in (
        (ds.encoding.get("source") or "").rsplit("/", 1)[-1],
        ds.attrs.get("dataset_name") or "",
    ):
        if not candidate:
            continue
        try:
            scan_start = _parse_scan_start_to_datetime(candidate)
            break
        except (ValueError, IndexError):
            continue
    if scan_start is not None:
        drift = abs(t - scan_start)
        if drift > _MAX_T_VS_FILENAME_DRIFT:
            raise ValueError(
                f"`t` coordinate ({t!s}) drifts from filename scan-start "
                f"({scan_start!s}) by {drift} — file appears mis-stamped."
            )
    return ds


def fill_missing_channels(ds: xr.Dataset) -> xr.Dataset:
    """Back-fill every missing per-channel variable.

    2-D imagery (``CMI_C*``, ``DQF_C*``) is filled with a fill-only
    ``ManifestArray`` synthesized from the hard-coded :data:`CMI_METADATA` /
    :data:`DQF_METADATA` so this works even when **zero** channels of a given
    prefix are present in the source file (e.g. an MCMIPF file produced
    without any DQF variables).

    Loaded per-band native variables are filled with a numpy array of the
    dtype's natural fill (``NaN`` for float, ``0`` for int) using the first
    existing channel as a template.
    """
    out = ds
    era = _era_for_ds(ds)
    metadata_table = (
        _PER_CHANNEL_2D_METADATA_POST if era == "post"
        else _PER_CHANNEL_2D_METADATA_PRE
    )

    for prefix in _PER_CHANNEL_2D:
        metadata = metadata_table[prefix]
        for i in ALL_CHANNELS:
            name = f"{prefix}{_ch(i)}"
            if name in out.variables:
                continue
            out = out.assign({
                name: xr.DataArray(
                    _make_fill_manifest_array(metadata),
                    dims=("y", "x"),
                    attrs={},
                )
            })

    for prefix in _PER_CHANNEL_BAND_LOADED:
        template_name = _first_present(out, prefix)
        if template_name is None:
            continue
        template_da = out[template_name]
        fill = _fill_for_dtype(template_da.dtype)
        for i in ALL_CHANNELS:
            name = f"{prefix}{_ch(i)}"
            if name in out.variables:
                continue
            out = out.assign({name: xr.full_like(template_da, fill)})

    for prefix in _PER_CHANNEL_SCALAR_LOADED:
        template_name = _first_present(out, prefix)
        if template_name is None:
            continue
        template_da = out[template_name]
        fill = _fill_for_dtype(template_da.dtype)
        for i in ALL_CHANNELS:
            name = f"{prefix}{_ch(i)}"
            if name in out.variables:
                continue
            out = out.assign({name: xr.full_like(template_da, fill)})

    return out


def fill_missing_scalar_metadata(ds: xr.Dataset) -> xr.Dataset:
    """Synthesize placeholders for any expected scalar/1-D metadata variable
    that's missing from this source file.

    Real-world NOAA files occasionally omit one of the telemetry scalars
    (e.g. ``percent_uncorrectable_GRB_errors``). The per-file schema is
    permissive (``required=False``), but the downstream ``xr.concat`` across
    a batch's files needs every variable to be present in every input — so
    if one file misses a variable, the whole batch's combine step fails.

    Walk :data:`_EXPECTED_SHARED_VARS` and assign a NaN-filled placeholder
    for any name not present. Fill values:
      - float dtypes → ``NaN``
      - ``datetime64`` → ``NaT``
      - int dtypes    → ``0`` (a sentinel value; the data was missing)
    """
    out = ds
    for name, dtype, dims in _EXPECTED_SHARED_VARS:
        if name in out.variables:
            continue
        # Determine shape — prefer the dataset's actual size for the dim,
        # falling back to the canonical _SHARED_DIM_SIZES table if the dim
        # isn't present in the dataset at all yet.
        shape = tuple(
            out.sizes[d] if d in out.sizes else _SHARED_DIM_SIZES[d]
            for d in dims
        )
        np_dtype = np.dtype(dtype)
        if np.issubdtype(np_dtype, np.floating):
            fill_val: Any = np.nan
        elif np.issubdtype(np_dtype, np.datetime64):
            fill_val = np.datetime64("NaT")
        else:
            fill_val = 0
        arr = np.full(shape, fill_val, dtype=np_dtype)
        out = out.assign({name: xr.DataArray(arr, dims=dims)})
    return out


def finalize_per_channel(ds: xr.Dataset) -> xr.Dataset:
    """Per-channel encoding fixups for the cleaned dataset.

    Unlike the old ``stack_channels``, this preserves every per-channel
    variable individually so each variable's CF attrs (long_name, valid_range,
    ...) stay verbatim from the source.

    What this still has to do:

    - Restore the int32 + ``_FillValue=-1`` encoding on every
      ``outlier_pixel_count_C{NN}``. CF decoding promotes the source int32
      (with ``_FillValue``) to float64 in memory so NaN can stand in for
      fill; ``fill_missing_channels`` then uses ``xr.full_like`` to synthesize
      missing channels, which drops the encoding. Without restoring it,
      xarray emits a SerializationWarning on write and can't round-trip the
      fill positions back to int storage.
    - Preserves x and y int16+scale_factor encoding verbatim from source.
      Previously stripped to silence a SerializationWarning; now kept so the
      round-trip dataset matches the source. See issue (4c) in
      REINGEST_PLAN.md.
    - For every variable that didn't have a ``_FillValue`` in its source attrs,
      explicitly sets ``encoding["_FillValue"] = None``. Without this, xarray's
      writer auto-adds ``_FillValue=nan`` to any float variable that lacks one,
      so reading the icechunk store back shows a fill value the source NetCDF
      never had. See issue (4d) in REINGEST_PLAN.md.
    - Casts ``scale_factor`` and ``add_offset`` (in attrs *and* encoding) back
      to float32 on every variable that carries them. VirtualiZarr's HDFParser
      surfaces these as Python ``float`` (i.e. float64), but the source NetCDF
      stores them as float32 — and the precision difference makes CMI decode
      to float64 instead of float32, with ~22M pixels per channel disagreeing
      at the last few mantissa bits. See issue (4a) in REINGEST_PLAN.md.
    - Backfills canonical CF attrs from :data:`_CANONICAL_VAR_ATTRS`
      (currently ``units='rad'`` on ``x_image_bounds`` / ``y_image_bounds``)
      via ``setdefault`` so any value already present in source is
      preserved. NOAA's earlier-era source files lacked these CF attrs;
      newer files added them. See GOES_DATA_INCONSISTENCIES.md.
    - Overrides canonical *expanded* CF attrs from
      :data:`_CANONICAL_VAR_ATTR_OVERRIDES` (currently ``valid_range=[0, 4]``
      on every ``DQF_C{NN}``) — these are written unconditionally so the
      icechunk-stored value is always the wider/modern range, regardless of
      what the source file had. The verifier accepts source-side narrower
      ranges as a subset of icechunk's. See GOES_DATA_INCONSISTENCIES.md.
    """
    out = ds

    # Restore the int32 encoding on every per-channel outlier_pixel_count_C{NN}.
    for i in ALL_CHANNELS:
        name = f"outlier_pixel_count_{_ch(i)}"
        if name in out.variables:
            out[name].encoding["dtype"] = np.int32
            out[name].encoding["_FillValue"] = np.int32(-1)

    # Suppress xarray's auto-added _FillValue=NaN on every *float-encoded*
    # variable that didn't carry one in the source. CF decoding lifts a real
    # _FillValue from attrs into .encoding, so absence here implies the source
    # didn't have one.
    #
    # Only do this for float encoded dtypes. For int-encoded variables (e.g.
    # x/y which are int16 with scale_factor on disk), setting _FillValue=None
    # makes Zarr v3 write `"fill_value": null`, which fails to round-trip —
    # xarray's int decoder rejects `None`. Int variables don't get the
    # auto-NaN treatment anyway, so leaving them alone is correct.
    for var in (*out.variables.values(),):
        if "_FillValue" in var.encoding:
            continue
        encoded_dtype = np.dtype(var.encoding.get("dtype", var.dtype))
        if np.issubdtype(encoded_dtype, np.floating):
            var.encoding["_FillValue"] = None

    # Cast scale_factor / add_offset back to float32 wherever they appear.
    # See issue (4a) above.
    for var in (*out.variables.values(),):
        for key in ("scale_factor", "add_offset"):
            if key in var.attrs:
                var.attrs[key] = np.float32(var.attrs[key])
            if key in var.encoding:
                var.encoding[key] = np.float32(var.encoding[key])

    # Backfill canonical CF attrs on variables where NOAA's earlier-era
    # source files lacked them (later files added them). Use setdefault so
    # we preserve any value that's already there; only add if absent. See
    # GOES_DATA_INCONSISTENCIES.md for which attrs we know drifted.
    for var_name, canonical_attrs in _CANONICAL_VAR_ATTRS.items():
        if var_name not in out.variables:
            continue
        for attr_name, attr_value in canonical_attrs.items():
            out[var_name].attrs.setdefault(attr_name, attr_value)

    # Apply canonical *overrides* — attrs NOAA monotonically expanded over
    # time. Unlike the setdefault block above, this *always* writes the
    # canonical (wider/modern) value, regardless of what source had. The
    # verifier accepts older narrower source values as a subset of the
    # icechunk-stored wider range.
    for var_name, override_attrs in _CANONICAL_VAR_ATTR_OVERRIDES.items():
        if var_name not in out.variables:
            continue
        for attr_name, attr_value in override_attrs.items():
            out[var_name].attrs[attr_name] = attr_value

    return out


def consolidate_band_coords(ds: xr.Dataset) -> xr.Dataset:
    """Collapse the 16 per-channel ``band_id_C{NN}`` / ``band_wavelength_C{NN}``
    coords into a single ``band`` dim coord + ``wavelength(band)`` non-dim coord
    of size 16.

    The source NetCDF carries those as 32 separate ``(band: 1,)`` coords —
    each ``band_id_C{NN}`` holds just the integer N, each
    ``band_wavelength_C{NN}`` holds one float. Information content is uniform
    in attrs and trivially compressible in values, so a single 16-element
    `band` axis is more useful than 32 degenerate singletons.

    The data variables (``CMI_C{NN}``, ``DQF_C{NN}``, stats, etc.) stay
    per-channel — they have meaningful per-channel CF attrs that band-
    stacking would clobber (see REINGEST_PLAN.md issue 2). This consolidation
    only touches the two trivial-attr coord families.

    The verifier (``mcmipf_verifier.verify_timestep``) is calibration-aware
    of this consolidation: it cross-checks each source ``band_id_C{NN}`` /
    ``band_wavelength_C{NN}`` value against the corresponding element of the
    consolidated ``band`` / ``wavelength`` arrays in icechunk.
    """
    out = ds

    # Harvest 16 scalar values from each per-channel coord. .values.flat[0]
    # handles both (band: 1,) shape from raw source and (t, band: 1,) shape
    # if this runs after add_time_dimension.
    band_ids: list[Any] = []
    wavelengths: list[Any] = []
    sample_band_id_attrs: dict[str, Any] = {}
    sample_wavelength_attrs: dict[str, Any] = {}
    for i in ALL_CHANNELS:
        bid_name = f"band_id_{_ch(i)}"
        bwl_name = f"band_wavelength_{_ch(i)}"
        if bid_name not in out.variables:
            raise ValueError(f"consolidate_band_coords: missing {bid_name!r}")
        if bwl_name not in out.variables:
            raise ValueError(f"consolidate_band_coords: missing {bwl_name!r}")
        band_ids.append(out[bid_name].values.flat[0])
        wavelengths.append(out[bwl_name].values.flat[0])
        if i == 1:
            sample_band_id_attrs = dict(out[bid_name].attrs)
            sample_wavelength_attrs = dict(out[bwl_name].attrs)

    # Drop the 32 per-channel coords. After this the ``band`` dim is orphaned
    # (no remaining variable references it).
    drop_names = [f"band_id_{_ch(i)}" for i in ALL_CHANNELS] + [
        f"band_wavelength_{_ch(i)}" for i in ALL_CHANNELS
    ]
    out = out.drop_vars(drop_names)

    # Build the consolidated coords. dtypes match the source per-channel
    # coords (int8 for band_id, float32 for band_wavelength).
    band_arr = np.array(band_ids, dtype=np.int8)
    wavelength_arr = np.array(wavelengths, dtype=np.float32)
    band_da = xr.DataArray(
        band_arr, dims="band", name="band", attrs=sample_band_id_attrs,
    )
    wavelength_da = xr.DataArray(
        wavelength_arr, dims="band", name="wavelength", attrs=sample_wavelength_attrs,
    )
    out = out.assign_coords(band=band_da, wavelength=wavelength_da)
    # Don't auto-add _FillValue=nan on float wavelength coord round-trip
    out["wavelength"].encoding["_FillValue"] = None
    return out


def add_time_dimension(ds: xr.Dataset) -> xr.Dataset:
    """``expand_dims('t')`` on every data variable using the scalar ``t`` coord.

    Defensively preserves ``.encoding`` across the ``expand_dims`` call so the
    per-channel ``outlier_pixel_count_C{NN}`` int32 + ``_FillValue`` encoding
    set in :func:`finalize_per_channel` survives into the final dataset.
    """
    if "t" not in ds.coords:
        raise ValueError("Expected a scalar 't' coordinate in the dataset.")
    t_value = ds["t"].values
    # Capture the source t's attrs + encoding before we drop it, so we can
    # re-attach them to the new t coord post-expansion. Without this xarray
    # silently loses t.attrs (long_name, standard_name, axis) and switches the
    # underlying dtype encoding from float64 → int64.
    t_attrs = dict(ds["t"].attrs)
    t_encoding = dict(ds["t"].encoding)

    out = ds.drop_vars("t")
    new_data_vars: dict[str, xr.DataArray] = {}
    for name, da in out.data_vars.items():
        expanded = da.expand_dims({"t": [t_value]}, axis=0)
        expanded.encoding = dict(da.encoding)
        new_data_vars[name] = expanded

    new_ds = xr.Dataset(new_data_vars, coords=out.coords)
    new_ds["t"].attrs.update(t_attrs)
    new_ds["t"].encoding.update(t_encoding)
    return new_ds


def validate_cleaned(ds: xr.Dataset) -> xr.Dataset:
    """Validate the cleaned dataset. Raises immediately on failure (pandera ``lazy=True``)."""
    make_cleaned_mcmipf_schema().validate(ds, lazy=True)
    return ds


# =============================================================================
# Top-level entry point
# =============================================================================
def preprocess(
    ds: xr.Dataset,
    *,
    loadable_variables: Iterable[str] = DEFAULT_LOADABLE_VARIABLES,
) -> xr.Dataset:
    """Run the full pipeline, raising immediately on the first failure.

    Suitable as the ``preprocess`` argument of
    :func:`virtualizarr.open_virtual_mfdataset`.

    Any exception from a stage is wrapped in :class:`MCMIPFValidationError` so
    the source filename is always attached; the original exception is preserved
    as ``__cause__`` and ``MCMIPFValidationError.original``.
    """
    try:
        validate_raw(ds, loadable_variables=loadable_variables)
        cleaned = fill_missing_channels(ds)
        cleaned = fill_missing_scalar_metadata(cleaned)
        cleaned = finalize_per_channel(cleaned)
        cleaned = consolidate_band_coords(cleaned)
        cleaned = add_time_dimension(cleaned)
        validate_cleaned(cleaned)
        return cleaned
    except Exception as e:
        raise MCMIPFValidationError(_source_of(ds), e) from e

## Smoke test

In [15]:
#!/usr/bin/env -S uv run
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "pandera[xarray]>=0.31.1",
#     "virtualizarr[hdf] @ git+https://github.com/zarr-developers/VirtualiZarr.git@main",
#     "obstore",
#     "obspec-utils",
#     "xarray",
# ]
# ///
"""Smoke test: run open_virtual_mfdataset + preprocess on a few real S3 files."""

from __future__ import annotations

import itertools
import os
import sys

import obstore as obs
import virtualizarr as vz
import xarray as xr
from obspec_utils.registry import ObjectStoreRegistry


def smoke_test(n_files: int = 3, *, start: str = CODEC_CHANGE_MARKER) -> xr.Dataset:
    """Open the first ``n_files`` from ``start`` via open_virtual_mfdataset + preprocess.

    Returns the combined virtual dataset. Raises ``MCMIPFValidationError`` on
    any file's pipeline failure (with the source filename attached).
    """
    urls = list(itertools.islice(iter_archive(start=start), n_files))
    if not urls:
        raise ValueError(f"No files found starting at {start!r}.")
    print(f"Opening {len(urls)} file(s):")
    for u in urls:
        print(f"  {u}")

    store = obs.store.from_url(BUCKET, region="us-east-1", skip_signature=True)
    registry = ObjectStoreRegistry({BUCKET: store})

    vmds = vz.open_virtual_mfdataset(
        urls,
        registry=registry,
        parser=vz.parsers.HDFParser(),
        loadable_variables=list(DEFAULT_LOADABLE_VARIABLES),
        preprocess=preprocess,
        concat_dim="t",
        combine="nested",
    )
    return vmds

In [16]:
%%time
vds = smoke_test(n_files=3)

In [17]:
vds

<xarray.Dataset> Size: 4GB
Dimensions:                                 (t: 3, number_of_time_bounds: 2,
                                             number_of_image_bounds: 2,
                                             y: 5424, x: 5424, band: 16)
Coordinates:
  * t                                       (t) datetime64[ns] 24B 2023-04-19...
  * y                                       (y) float64 43kB 0.1518 ... -0.1518
  * x                                       (x) float64 43kB -0.1518 ... 0.1518
  * band                                    (band) int8 16B 1 2 3 4 ... 14 15 16
    wavelength                              (band) float32 64B 0.47 ... 13.27
    y_image                                 float32 4B 0.0
    x_image                                 float32 4B 0.0
Dimensions without coordinates: number_of_time_bounds, number_of_image_bounds
Data variables: (12/188)
    time_bounds                             (t, number_of_time_bounds) datetime64[ns] 48B ...
    goes_imager_project

In [18]:
print(vds[["CMI_C01", "DQF_C01"]])

## Set up repo

In [19]:
al_client = al.Client()
al_client.login()

Successfully refreshed tokens! Token stored at /home/cloudenv/.arraylake/token.json


╭───────────────────────────────────────────────── User Details ──────────────────────────────────────────────────╮
│ Name: Tom Nicholas                                                                                              │
│ Email: tom@earthmover.io                                                                                        │
│ Id: 0a632e93-b8ec-4114-b352-d4ba6615c826                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Each data variable at each time step has 24x24=576 chunks

Our plan for ingestion is to:
- Commit once per 15 days of data (~1500 files)
  - This will allow us to use ~1000 dask workers in parallel (the most my AWS limits will allow me)
  - It's then ~1 million chunks per data variable per commit
  - All the other variables have 576x fewer chunks so are basically irrelevant, and we just load them directly.
  - That's a solid number of references to commit in one go (~30 million), but it's also not too many to upload during a single commit (<50 million, a hard cap in Icechunk).
- Split the manifests:
  - Constraints:
      - We never make a single manifest > 2GiB in size, as that causes an error (this threshold seems to be passed at ~15M virtual chunk refs).
      - As manifests increase in size along the "t" dimension in subsequent commits, we want to avoid too much pain from O(n) slowdowns in commit runtime from rewriting previous not-yet-split manifests. We avoid this by splitting along "t" fairly frequently - ideally as frequently as we commit.
      - We split CMI across bands - this is inherent in the fact that each band is a separate variable (needed because the scale_factor and offset encoding differ per-band).
      - We attempt to split CMI across spatial dimensions too, to make point timeseries querying faster, by needing to fetch fewer manifests to read data from a single spatial point.
      - But we don't want too many manifests per array either - there is a linear search on chunk coords, so we pay a lookup cost for many on every single chunk fetch. 100k is about the upper limit.
  - Choices:
      - roughly once per 15 days
      - We're aiming for roughly ~11-M chunks per manifest, and as each timestep has ~1.2M chunks per variable this would be ~6M chunks per manifest.
      - We do this because we don't want too many manifests as this makes searching them slow, but we also don't want too big manifests as this makes fetching them slow.
      - The would put the total number of manifests at 7 years * 365 days per year / 15 days per manifest = ~100 Manifests per variable.
      - Also it seems plausible that a user wants to examine all the data for a given day or month, and this means that workload requires only fetching ~6 manifests per variable.
      - (This strategy is less suitable for timeseries analysis, but the underlying chunking is already not ideal for that, and we suspect people want to fetch whole images rather than specific points anyway.)
      - We probably only need to bother splitting for the data variables `CMI` and `DQF`, as it would take 576 / 10 / 12 = 4.8 years of data before the manifests of the non-spatially-dependent variables become the same size as these spatially-dependent ones.
- Avoid making the chunks of the time-dependent coordinate variables too small
  - Because we will load these into memory using VZ, and we are committing once per day, each chunk of e.g. the `time` variable would by default be ~100 elements long in the time dimension, meaning there would be 7 years * 365 days = 2555 miniscule chunks making up that coordinate variable.
  - Instead if we tell Zarr we want the chunk lengths along `time` for these variables to be ~20000 elements long, we will have ~100 chunks of 0.16MB size, which is a bit more reasonable.

In [20]:
split_config = ic.ManifestSplittingConfig.from_dict(
    {
        ic.ManifestSplitCondition.name_matches("CMI"): {
          ic.ManifestSplitDimCondition.DimensionName("y"): 12,   # 2 splits (y=24 chunks)
          ic.ManifestSplitDimCondition.DimensionName("x"): 12,   # 2 splits (x=24 chunks)
          ic.ManifestSplitDimCondition.DimensionName("t"): 15 * 144,  # ~1 new manifest per 15 days of data
        },
        ic.ManifestSplitCondition.name_matches("DQF"): {
          ic.ManifestSplitDimCondition.DimensionName("y"): 12,   # 2 splits (y=24 chunks)
          ic.ManifestSplitDimCondition.DimensionName("x"): 12,   # 2 splits (x=24 chunks)
          ic.ManifestSplitDimCondition.DimensionName("t"): 15 * 144,  # ~1 new manifest per 15 days of data
        },
    }
)

In [42]:
# we want the manifest splitting to be permanently attached to this repo, so it definitely gets used by any writes
ic_repo = al_client.create_repo(
    name="earthmover-public/goes-16",
    bucket_config_nickname="earthmover",  # bucket in us-east-1
    authorize_virtual_chunk_access={"s3://noaa-goes16/": "noaa-goes16"},
    config=ic.RepositoryConfig(
        manifest=ic.ManifestConfig(splitting=split_config),
    ),
)

In [43]:
# authorize Arraylake repos to read contents from the AWS open data bucket
al_client.set_virtual_chunk_access_policy(
    org="earthmover-public",
    bucket_nickname="noaa-goes16",
    subprefix="",
    public=True,
)

ExplicitVirtualChunkAccessPolicyResponse(bucket_id='57e6fbd5-c231-4b33-98d2-2ad62912f044', subprefix='', public=True, url_prefix='s3://noaa-goes16/', created=datetime.datetime(2026, 5, 19, 17, 26, 47), created_by='0a632e93-b8ec-4114-b352-d4ba6615c826')

In [49]:
# nuke the repo back to it's starting point
# ic_repo.reset_branch(
#     branch="v2-ingest",
#     snapshot_id="1CECHNKREP0F1RSTCMT0",   # universal icechunk initial-empty snapshot
#     #from_snapshot_id=ic_repo.lookup_branch("v2-ingest"),  # safety 
# )

In [46]:
# reset the repo's config
ic_repo = ic_repo.reopen(
    config=ic.RepositoryConfig(
        manifest=ic.ManifestConfig(splitting=split_config),
    ),
)
ic_repo.save_config()

## Ingestion

In [21]:
# For ingestion jobs we want to manage the cache to evict snapshots and manifests as soon as they have been written, to keep memory usage as low as possible
# but we don't want readers to see this config, which is why we don't set it at repo creation time.
caching_config = ic.config.CachingConfig(
    # just keep the current and previous snapshot
    num_snapshot_nodes=2,
    # over 2x the total number of refs in one commit, but low enough to not accumulate lots of useless refs across commits 
    num_chunk_refs=50_000_000,
)

In [22]:
ic_repo = al_client.get_repo(
    name="earthmover-public/goes-16",
    config=ic.RepositoryConfig(
        manifest=ic.ManifestConfig(splitting=split_config),
        caching=caching_config,
    ),
)

In [23]:
ic_repo.config

icechunk.config.RepositoryConfig(
    inline_chunk_threshold_bytes=None,
    get_partial_values_concurrency=None,
    max_concurrent_requests=None,
    num_updates_per_repo_info_file=None,
    compression=None,
    caching=icechunk.config.CachingConfig(
        num_snapshot_nodes=2,
        num_chunk_refs=50000000,
        num_transaction_changes=None,
        num_bytes_attributes=None,
        num_bytes_chunks=None,
    ),
    storage=icechunk.storage.StorageSettings(
        unsafe_use_conditional_create=None,
        unsafe_use_conditional_update=None,
        unsafe_use_metadata=None,
        storage_class=None,
        metadata_storage_class=None,
        chunks_storage_class=None,
        minimum_size_for_multipart_upload=None,
        concurrency=None,
        retries=None,
        timeouts=None,
    ),
    manifest=icechunk.config.ManifestConfig(
        preload=None,
        splitting=icechunk.config.ManifestSplittingConfig(
            split_sizes=[(icechunk.config.ManifestSp

In [25]:
cluster = coiled.Cluster(
    name="GOES",
    n_workers=[1000, 1500],  # roughly highest I can do without an AWS account quota increase - sad
    region="us-east-1",
    #worker_vm_types="m8g.medium",
    worker_vm_types=[
        # spread across c6/c7/c8 + m6/m7/m8, two sizes each = ~18 spot pools
        "t4g.small", "t4g.medium",# "t4g.large",
        "c6g.medium",
        "m6g.medium",
        "c7g.medium",
        "m7g.medium",
        "c8g.medium",
        "m8g.medium",
        #"c6g.large", "c7g.large", "c8g.large",
        #"m6g.large", "m7g.large", "m8g.large",
        #"c6g.xlarge", "c7g.xlarge", "c8g.xlarge",
    ],
    scheduler_vm_types=["c8g.2xlarge", "m7g.2xlarge", "m8g.2xlarge"],  # (one time I hit AWS capacity constraints so let Coiled choose)
    spot_policy="spot",  # important to avoid 10x cost spikes!
    #use_best_zone=True,
    allow_cross_zone=True,
    wait_for_workers=False,
)
client = cluster.get_client()
client

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   virtualizarr │ Wheel built from                                            │
│                │ git+https://github.com/zarr-developers/VirtualiZarr.git@c   │
│                │ 95bc2726eea4610f8420da3e8441ee9b5833cc7                     │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│             ╷                                                    ╷           │
│   Package   │ Error                                              │ Level     │
│ ╶───────────┼────────────────────────────────────────────────────┼─────────╴ │
│   shiboken6 │ cannot find shiboken6~=6.11.1 on pypi.org. If you  │ Warning   │
│             │ are using a custom PyPI URL, please see            │           │
│             │ https://docs.coiled.io/user_guide/software/packag… │           │
│             │ for more instructions.                             │           │
│             ╵                                                    ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯


<Client: 'tls://10.0.214.222:8786' processes=0 threads=0, memory=0 B>

Define orchestration function for parsing and concatenating many files at once then committing. Has a bunch of helpers for idempotent resumability and sanity checks.

In [26]:
"""Batched ingest of GOES-16 MCMIPF into Icechunk.

Provides:
  * :func:`timer`             — context manager that prints elapsed time on exit.
  * :func:`days_to_ingest`    — list of ``((year, doy), urls)`` tuples to iterate.
  * :func:`ingest_all_days`   — the outer loop. Groups days into batches
                               (``batch_size`` per commit; default 15) and runs
                               per-batch open/validate/write/commit, with
                               per-step timing, auto-resume, and per-batch
                               memory cleanup.

``loop_start_day`` / ``loop_end_day`` restrict the run to a sub-range of the
listed days; ``resume=True`` (default) auto-skips any days already committed
to ``repo`` so re-runs are idempotent.
"""

from __future__ import annotations

import contextlib
import datetime
import gc
import os
import sys
import time
import warnings
from collections.abc import Iterable
from typing import TYPE_CHECKING

import numpy as np
import obstore as obs
import pandas as pd
import virtualizarr as vz
from obspec_utils.registry import ObjectStoreRegistry


# Calendar date of the earliest day present in the GOES-16 ABI-L2-MCMIPF archive
# (computed once as DOY 59 of 2017 by listing the bucket; hard-coded here so the
# loop doesn't re-query S3 on every run).
ARCHIVE_START_DATE = datetime.date(2017, 2, 28)

if TYPE_CHECKING:
    import icechunk


@contextlib.contextmanager
def timer(name: str):
    """Print elapsed time for a labelled block when the block exits."""
    t0 = time.perf_counter()
    yield
    print(f"  {name}: {time.perf_counter() - t0:.1f}s", flush=True)


def _date_from_doy(year: int, doy: int) -> datetime.date:
    """Convert ``(year, day-of-year)`` to a ``datetime.date``."""
    return datetime.date(year, 1, 1) + datetime.timedelta(days=doy - 1)





def _archive_day_index(year: int, doy: int) -> int:
    """Calendar days from :data:`ARCHIVE_START_DATE` to the given (year, doy).

    Day 0 is the first day of the archive (2017-02-28). For 2023-04-19 (the
    codec-change marker) this returns ~2241.
    """
    return (_date_from_doy(year, doy) - ARCHIVE_START_DATE).days


def _schema_exists(
    repo: "icechunk.Repository",
    branch: str,
    group: str | None = None,
) -> bool:
    """Return ``True`` if a zarr group is already committed at ``group`` on ``branch``."""
    import zarr

    try:
        session = repo.readonly_session(branch=branch)
        zarr.open_group(store=session.store, path=group or "", zarr_format=3, mode="r")
        return True
    except FileNotFoundError:
        return False


def _last_committed_day(
    repo: "icechunk.Repository",
    branch: str,
    group: str | None = None,
) -> tuple[tuple[int, int], np.datetime64] | None:
    """Return ``((year, doy), last_t)`` of the latest already-committed day on
    ``branch``, or ``None`` if the group doesn't exist or has no ``t`` data yet.

    Verifies that the existing ``t`` index is strictly increasing before
    trusting it; raises :class:`ValueError` if it isn't (rather than silently
    auto-resuming on top of a corrupted dataset). When the invariant holds,
    returns the day corresponding to the **final** ``t`` value directly
    (cheaper than ``.max()`` and equivalent given monotonicity), plus the
    actual final ``t`` value for cross-batch boundary checks.
    """
    import xarray as xr

    try:
        session = repo.readonly_session(branch=branch)
        existing = xr.open_zarr(
            session.store, group=group or None, chunks=None, zarr_format=3,
        )
    except (FileNotFoundError, KeyError):
        return None
    if "t" not in existing.coords or existing.sizes.get("t", 0) == 0:
        return None

    t_idx = existing.xindexes["t"].to_pandas_index()
    if not (t_idx.is_monotonic_increasing and t_idx.is_unique):
        raise ValueError(
            "Cannot auto-resume: existing dataset's `t` index is not strictly "
            f"increasing (monotonic={t_idx.is_monotonic_increasing}, "
            f"unique={t_idx.is_unique}). Repair the existing data (or pass "
            "`resume=False` and write to a fresh group) before continuing."
        )

    last_t_np = existing["t"].values[-1]
    last_t_ts = pd.Timestamp(last_t_np)
    return (last_t_ts.year, int(last_t_ts.dayofyear)), np.datetime64(last_t_np, "ns")


def days_to_ingest(
    *,
    start: str | None = CODEC_CHANGE_MARKER,
    end: str | None = None,
) -> list[tuple[tuple[int, int], list[str]]]:
    """Eagerly enumerate the days/URLs we'll iterate. Suitable for slicing
    (``all_days[loop_start:loop_end]``)."""
    return list(iter_archive_by_day(start=start, end=end))


def _run_batch_verifications(
    *,
    repo: "icechunk.Repository",
    branch: str,
    group: str | None,
    registry: ObjectStoreRegistry | None,
    sample_urls: list[tuple[int, str]],
    zarr_async_concurrency: int | None = None,
) -> None:
    """Verify all sampled files for this batch — serial on the driver.

    Each verify_timestep runs on the driver (not wrapped in dask.delayed) so
    the materialised arrays land in the notebook process's memory, not on the
    small Coiled workers. The cluster is still used: verify_timestep opens the
    icechunk side with one-dask-block-per-variable chunking and ``.load()``s
    it, which dispatches the per-variable block fetches to workers and gathers
    them back to the driver.

    Any failure aborts the batch with the failing scan's full mismatch report.
    """
    n = len(sample_urls)
    with timer(f"Verifying {n} file(s) on driver"):
        for global_idx, url in sample_urls:
            print(
                f"  verifying file {global_idx} ({url.rsplit('/', 1)[-1]})",
                flush=True,
            )
            try:
                verify_timestep(
                    repo=repo, branch=branch, group=group,
                    source_url=url, registry=registry, verbose=False,
                    zarr_async_concurrency=zarr_async_concurrency,
                )
            except VerificationError:
                # Real data-mismatch — propagate immediately, no point retrying.
                raise
            except Exception as e:
                # Anything else (IcechunkError from a worker with stale
                # credentials, transient S3 hiccup, etc.) — retry once in
                # synchronous-on-driver mode so the worker dispatch path is
                # avoided. If THAT also fails, propagate the new exception.
                print(
                    f"  ⚠ first verify attempt failed "
                    f"({type(e).__name__}): {e!s:.300}",
                    flush=True,
                )
                print(
                    "  retrying synchronously on driver "
                    "(force_synchronous=True)...",
                    flush=True,
                )
                verify_timestep(
                    repo=repo, branch=branch, group=group,
                    source_url=url, registry=registry, verbose=False,
                    zarr_async_concurrency=zarr_async_concurrency,
                    force_synchronous=True,
                )


def ingest_all_days(
    repo: "icechunk.Repository",
    *,
    branch: str = "main",
    group: str | None = None,
    registry: ObjectStoreRegistry | None = None,
    loop_start_day: int = 0,
    loop_end_day: int | None = None,
    batch_size: int = 15,
    start: str | None = CODEC_CHANGE_MARKER,
    end: str | None = None,
    loadable_variables: Iterable[str] = DEFAULT_LOADABLE_VARIABLES,
    resume: bool = True,
    verify_every: int | None = None,
    zarr_async_concurrency: int | None = None,
) -> None:
    """Ingest every selected day into ``repo``, in batches of ``batch_size``
    days per commit.

    The day list is produced in two stages, controlled by two pairs of arguments:

    1. **URL bounds** (``start`` / ``end``) — file URLs that filter *which days
       get listed from S3*. They're forwarded to
       :func:`iter_archive_by_day(start=..., end=...)`. Default ``start`` is
       :data:`CODEC_CHANGE_MARKER` so only days at or after the codec change
       are considered.
    2. **Index slice** (``loop_start_day`` / ``loop_end_day``) — integer indices
       into the list produced in step 1 (equivalent to
       ``all_days[loop_start_day:loop_end_day]``). Use this for restricting a
       single run to a sub-range.

    Within that range, days are processed in **batches of ``batch_size``
    (default 15) days per commit**. Each batch concatenates all its files into
    one virtual dataset and writes a single icechunk commit. Tail batches with
    fewer than ``batch_size`` days are committed normally.

    Example::

        # First run: ingest from the codec change forward, 15 days/commit.
        ingest_all_days(
            repo=ic_repo,
            group=f"{PRODUCT}/post-2023-04-19",
            start=CODEC_CHANGE_MARKER,
        )

        # Re-run later — auto-resume picks up where the last commit ended.
        # Same call works idempotently.
        ingest_all_days(
            repo=ic_repo,
            group=f"{PRODUCT}/post-2023-04-19",
            start=CODEC_CHANGE_MARKER,
        )

    Other parameters:

    - ``repo`` — an already-opened :class:`icechunk.Repository`.
    - ``branch`` — the branch to write to (default ``"main"``).
    - ``group`` — group path inside the store to write into; ``None`` means the
      root group. Forwarded to ``vds.vz.to_icechunk(..., group=group)`` and used
      by the schema-existence check.
    - ``batch_size`` — days per commit (default 15). Each batch's combined
      virtual dataset has ``batch_size × ~144`` files; commit produces a single
      icechunk snapshot covering those days.
    - ``registry`` — an :class:`ObjectStoreRegistry`; defaults to anonymous
      ``noaa-goes16`` on ``us-east-1``.
    - ``loadable_variables`` — passed through to
      :func:`virtualizarr.open_virtual_mfdataset`.
    - ``verify_every`` — if set, every Nth ingested file is round-trip-
      verified against its source NetCDF via :func:`mcmipf_verifier.verify_timestep`
      after the enclosing batch commits. A single mismatch aborts the loop
      with the full per-variable report. ``None`` (default) disables. Count is
      cumulative across batches within one run but resets on a new
      :func:`ingest_all_days` invocation; resuming from an existing dataset
      starts the count over. Recommended values to keep verification cost
      bounded: ``1000`` (≈ 1 verify per 7 default batches) or
      ``batch_size * 144`` (≈ 1 verify per commit).
    - ``zarr_async_concurrency`` — if set, calls
      ``zarr.config.set({"async.concurrency": N})`` once on entry so zarr's
      icechunk-backed reads/writes can fan out wider. Recommended: ``16`` for
      small / slow-connection machines, ``64`` for high-performance cloud
      workers. The value is also forwarded into each delayed
      ``verify_timestep`` so dask workers set their local zarr config on
      their first verification. ``None`` (default) leaves the global config
      untouched.

    Behaviour notes:

    - **First-write detection is automatic.** If no root zarr group exists on
      ``branch``, the first batch writes without ``append_dim`` to create the
      schema; every subsequent batch appends along ``t``.
    - **Auto-resume (default ``resume=True``).** Before iterating, the existing
      ``t`` index is verified strictly increasing and its final value is used
      to drop any selected day at or before the last committed one. Re-running
      with overlapping bounds is therefore idempotent. A corrupted existing
      ``t`` axis raises ``ValueError`` at startup. Pass ``resume=False`` to
      ignore the existing dataset (e.g. when writing to a fresh group).
    - **One commit per batch.** Commit message format::

        Ingested all GOES-16 ABI-L2-MCMIPF files for YYYY-MM-DD through YYYY-MM-DD.
        Archive days N1-N2.

      where ``N1`` / ``N2`` are the calendar-day offsets from
      :data:`ARCHIVE_START_DATE` (2017-02-28 = day 0).
    - **Per-step timing** is printed via :func:`timer` as each step completes
      (open / monotonic-check / open-session / write / commit / cleanup).
    - **Memory is explicitly released** after each batch
      (``vds.close()``, ``del vds``, ``del session``, ``gc.collect()``).
    """
    if registry is None:
        store = obs.store.from_url(BUCKET, region="us-east-1", skip_signature=True)
        registry = ObjectStoreRegistry({BUCKET: store})

    if zarr_async_concurrency is not None:
        import zarr
        zarr.config.set({"async.concurrency": zarr_async_concurrency})
        print(
            f"zarr async concurrency set to {zarr_async_concurrency} "
            f"(process-global; affects write path on the driver and verify "
            f"reads on workers if forwarded via dask.delayed).",
            flush=True,
        )

    parser = vz.parsers.HDFParser()
    loadable_variables = list(loadable_variables)

    all_days_to_ingest = days_to_ingest(start=start, end=end)
    end_idx = loop_end_day if loop_end_day is not None else len(all_days_to_ingest)
    selected = all_days_to_ingest[loop_start_day:end_idx]
    print(
        f"Ingesting {len(selected)} day(s) "
        f"[{loop_start_day}:{end_idx}] of {len(all_days_to_ingest)} total available.",
        flush=True,
    )

    is_first_write = not _schema_exists(repo, branch, group)
    if is_first_write:
        where = f"group {group!r}" if group else "root group"
        print(
            f"No existing schema on '{branch}' at {where} — the first iteration "
            "will create it (no append_dim).",
            flush=True,
        )

    # Auto-resume: query the existing dataset's last t value and skip every
    # already-committed day. Makes re-runs idempotent — rerunning with an
    # overlapping `loop_start_day` no longer double-ingests.
    skip_through: tuple[int, int] | None = None
    last_committed_t: np.datetime64 | None = None
    if resume and not is_first_write:
        info = _last_committed_day(repo, branch, group)
        if info is not None:
            skip_through, last_committed_t = info
            print(
                f"Auto-resuming: last committed day is "
                f"{skip_through[0]}-{skip_through[1]:03d} "
                f"(last t = {last_committed_t}). Skipping any selected days "
                f"at or before that.",
                flush=True,
            )

    n_batches = (len(selected) + batch_size - 1) // batch_size
    boundary_checked = False  # one-time cross-batch boundary assertion below
    files_ingested = 0        # cumulative count across batches (this run only)
    for batch_ind in range(n_batches):
        batch_start = batch_ind * batch_size
        batch = selected[batch_start : batch_start + batch_size]

        # Apply auto-resume by dropping days at or before the last committed day.
        # Partial batches at the resume boundary are handled here — a batch that
        # spans the boundary keeps only its uncommitted tail.
        if skip_through is not None:
            batch = [item for item in batch if item[0] > skip_through]
        if not batch:
            continue

        urls_for_this_batch = [u for _, urls in batch for u in urls]

        # One-time cross-batch boundary check: confirm the first uncommitted
        # file's filename-derived scan-start time is strictly after the last
        # committed `t`. Catches the case where iter_days / list_day_files
        # could ever return an out-of-order day list relative to existing
        # committed data — a path the per-batch monotonic check below cannot
        # see because it only inspects within-batch t values.
        if not boundary_checked and last_committed_t is not None:
            first_file_scan_start = parse_scan_start_to_datetime(urls_for_this_batch[0])
            if first_file_scan_start <= last_committed_t:
                raise ValueError(
                    f"Cross-batch ordering violation: existing data's last `t` "
                    f"is {last_committed_t}, but the first uncommitted file's "
                    f"scan-start ({urls_for_this_batch[0].rsplit('/', 1)[-1]}) "
                    f"parses to {first_file_scan_start}. The day list appears "
                    f"out of order relative to committed data."
                )
            boundary_checked = True
        (first_year, first_doy), _ = batch[0]
        (last_year, last_doy), _ = batch[-1]
        first_date = _date_from_doy(first_year, first_doy)
        last_date = _date_from_doy(last_year, last_doy)
        first_archive_day = _archive_day_index(first_year, first_doy)
        last_archive_day = _archive_day_index(last_year, last_doy)

        print(f"\n====== Batch iteration {batch_ind} ======")
        print(
            f"Ingesting {len(batch)} day(s) "
            f"({first_date.isoformat()} through {last_date.isoformat()}, "
            f"archive days {first_archive_day}-{last_archive_day})"
        )
        print(f"Total files in this batch: {len(urls_for_this_batch)}")

        with timer("Creating the virtual dataset"):
            vds = vz.open_virtual_mfdataset(
                urls_for_this_batch,
                registry=registry,
                parser=parser,
                preprocess=preprocess,
                # combine="nested" + concat_dim="t" preserves URL order (which
                # mcmipf_archive.list_day_files guarantees is chronological by
                # scan-start time). Using "by_coords" would sort by the cleaned
                # `t` value silently — masking any out-of-order file rather than
                # letting the monotonic check below catch it.
                combine="nested",
                concat_dim="t",
                combine_attrs="drop_conflicts",
                coords="minimal",
                compat="override",
                loadable_variables=loadable_variables,
                parallel="dask",
            )

        with timer("Checking t is strictly increasing"):
            # is_monotonic_increasing alone permits duplicate t values (e.g. a
            # reprocessed file), so combine with is_unique to assert strict.
            t_idx = vds.indexes["t"]
            if not (t_idx.is_monotonic_increasing and t_idx.is_unique):
                raise ValueError(
                    f"Batch {first_date.isoformat()}..{last_date.isoformat()}: "
                    f"`t` is not strictly increasing across the "
                    f"{len(urls_for_this_batch)} files "
                    f"(monotonic={t_idx.is_monotonic_increasing}, unique={t_idx.is_unique}). "
                    f"First 3 t values: {list(t_idx[:3])} ... last 3: {list(t_idx[-3:])}"
                )

        with timer("Opening an icechunk session"):
            session = repo.writable_session(branch)

        with timer("Writing to icechunk"):
            # xarray's CF encoder emits a SerializationWarning whenever it's
            # asked to write float in-memory data into an integer storage dtype
            # without a _FillValue. That's our x / y coords by design: the
            # source NetCDF stores them as int16 + scale_factor with no fill
            # (they're well-defined projection coords with no NaN possible) and
            # we preserve that encoding for round-trip fidelity with the source.
            # The warning is cosmetic; suppress it narrowly here so genuinely
            # unexpected SerializationWarnings still surface.
            from xarray.coding.variables import SerializationWarning
            with warnings.catch_warnings():
                warnings.filterwarnings(
                    "ignore",
                    category=SerializationWarning,
                    message=r".*floating point data as an integer dtype without any _FillValue.*",
                )
                if is_first_write:
                    vds.vz.to_icechunk(session.store, group=group)
                    is_first_write = False
                else:
                    vds.vz.to_icechunk(session.store, group=group, append_dim="t")

        with timer("Committing to icechunk"):
            msg = (
                f"Ingested all GOES-16 ABI-L2-MCMIPF files for "
                f"{first_date.isoformat()} through {last_date.isoformat()}. "
                f"Archive days {first_archive_day}-{last_archive_day}."
            )
            session.commit(msg)
            print(msg)

        with timer("Cleaning up memory and garbage collection"):
            vds.close()
            del vds
            del session
            gc.collect()

        # Post-commit verification — sample the files at global indices that
        # are multiples of `verify_every`, open them via mcmipf_verifier, and
        # assert their round-trip against the source NetCDF agrees on every
        # variable. A single mismatch aborts the loop with the full report.
        if verify_every is not None:
            sample_urls: list[tuple[int, str]] = []
            for i, url in enumerate(urls_for_this_batch):
                global_idx = files_ingested + i + 1  # 1-indexed
                if global_idx % verify_every == 0:
                    sample_urls.append((global_idx, url))
            if sample_urls:
                _run_batch_verifications(
                    repo=repo, branch=branch, group=group,
                    registry=registry, sample_urls=sample_urls,
                    zarr_async_concurrency=zarr_async_concurrency,
                )

        files_ingested += len(urls_for_this_batch)


# Era layout — keep in sync with the boundaries in mcmipf_archive. Each entry
# covers a (codec era × calibration era) combination that's internally
# consistent for xr.concat. See the era-boundary docstring in mcmipf_archive.
#   suffix             start                       end
_ERAS: tuple[tuple[str, str | None, str | None], ...] = (
    ("pre-2017-07-24",  None,                     LAST_EARLY_PREOP_FILE),
    ("pre-2017-10-31",  FIRST_LATE_PREOP_FILE,    LAST_PRE_OPERATIONAL_FILE),
    ("pre-2023-04-19",  FIRST_OPERATIONAL_FILE,   LAST_PRE_CODEC_CHANGE_FILE),
    ("post-2023-04-19", CODEC_CHANGE_MARKER,      None),
)


def ingest_all_eras(
    repo: "icechunk.Repository",
    *,
    base_group: str = "ABI-L2-MCMIPF",
    branch: str = "main",
    registry: ObjectStoreRegistry | None = None,
    batch_size: int = 15,
    loadable_variables: Iterable[str] = DEFAULT_LOADABLE_VARIABLES,
    resume: bool = True,
    eras: Iterable[str] | None = None,
    dry_run: bool = False,
    verify_every: int | None = None,
    zarr_async_concurrency: int | None = None,
) -> None:
    """Ingest the entire GOES-16 ABI-L2-MCMIPF archive into four sibling groups.

    Iterates the four era windows documented on
    :data:`mcmipf_archive.LATE_PREOP_BOUNDARY_DATETIME`,
    :data:`mcmipf_archive.OPERATIONAL_BOUNDARY_DATETIME`, and
    :data:`mcmipf_archive.CODEC_CHANGE_DATETIME`, and calls
    :func:`ingest_all_days` once per window. Each call writes into
    ``{base_group}/{era_suffix}`` (e.g. ``ABI-L2-MCMIPF/post-2023-04-19``) with
    URL bounds clipped to that window.

    Per-era ``ingest_all_days`` behaviour is unchanged: auto-resume, per-batch
    monotonic-t check, cross-batch boundary check, per-step timing. Within an
    era the calibration and codec pipeline are consistent, so virtualizarr's
    nested concat over ``t`` is safe.

    Parameters
    ----------
    repo, branch, registry, batch_size, loadable_variables, resume
        Forwarded to :func:`ingest_all_days`.
    base_group : str
        Parent group; each era is written into ``{base_group}/{suffix}``.
    eras : iterable of str, optional
        Restrict to a subset of era suffixes (``"pre-2017-07-24"``,
        ``"pre-2017-10-31"``, ``"pre-2023-04-19"``, ``"post-2023-04-19"``).
        Default is all four.
    dry_run : bool
        If True, print the per-era plan (group / start / end) and exit without
        opening any sessions. Useful for sanity checks before a full run.
    verify_every : int, optional
        Forwarded to :func:`ingest_all_days`. If set, every Nth ingested file
        within each era is round-trip-verified against its source NetCDF
        after its batch commits; the count resets at each era boundary.
    zarr_async_concurrency : int, optional
        Forwarded to :func:`ingest_all_days`. Sets zarr's ``async.concurrency``
        config once at the start of each per-era run; recommended ``64`` for
        cloud workers, ``16`` for slower connections.
    """
    selected_eras = (
        [e for e in _ERAS if e[0] in set(eras)] if eras is not None else list(_ERAS)
    )
    if not selected_eras:
        raise ValueError(f"No matching eras selected from {[e[0] for e in _ERAS]!r}")

    print(f"Ingest plan ({len(selected_eras)} era window(s)):", flush=True)
    for suffix, start, end in selected_eras:
        group = f"{base_group}/{suffix}"
        start_fname = start.rsplit("/", 1)[-1] if start else "(archive start)"
        end_fname = end.rsplit("/", 1)[-1] if end else "(archive end)"
        print(f"  era {suffix!r:18s} → group {group!r}")
        print(f"    start: {start_fname}")
        print(f"    end:   {end_fname}")
    if dry_run:
        print("\ndry_run=True — exiting without ingesting.", flush=True)
        return

    for suffix, start, end in selected_eras:
        group = f"{base_group}/{suffix}"
        print(f"\n############# Era {suffix!r} → {group!r} #############\n", flush=True)
        ingest_all_days(
            repo,
            branch=branch,
            group=group,
            registry=registry,
            batch_size=batch_size,
            start=start,
            end=end,
            loadable_variables=loadable_variables,
            resume=resume,
            verify_every=verify_every,
            zarr_async_concurrency=zarr_async_concurrency,
        )

In [83]:
# ingest the first of 4 eras
ingest_all_days(
    ic_repo,
    branch="v2-ingest",
    group="ABI-L2-MCMIPF/pre-2017-07-24",
    start=None,
    end=LAST_EARLY_PREOP_FILE,
    #loop_end_day=5,                # just the first 5 days
    verify_every=1000,
    zarr_async_concurrency=64,
)

In [84]:
ingest_all_days(
    ic_repo,
    branch="v2-ingest",
    group="ABI-L2-MCMIPF/pre-2017-10-31",     # ← new group
    start=FIRST_LATE_PREOP_FILE,               # ← new lower bound
    end=LAST_PRE_OPERATIONAL_FILE,             # ← new upper bound
    verify_every=1000,
    zarr_async_concurrency=64,
)

IcechunkError:   x object store error dispatch failure
  | 
  | context:
  |    0: icechunk::asset_manager::fetch_snapshot
  |            with snapshot_id=YRJCHHM89E733P9YVSB0
  |              at icechunk/src/asset_manager.rs:1279
  |    1: icechunk::session::get_node
  |            with path=/ABI-L2-MCMIPF/pre-2017-10-31/std_dev_reflectance_factor_C09
  |              at icechunk/src/session.rs:1109
  |    2: icechunk::session::get_chunk_ref
  |            with path=/ABI-L2-MCMIPF/pre-2017-10-31/std_dev_reflectance_factor_C09 coords=ChunkIndices([3])
  |              at icechunk/src/session.rs:1154
  |    3: icechunk::session::get_chunk_reader
  |            with path=/ABI-L2-MCMIPF/pre-2017-10-31/std_dev_reflectance_factor_C09 coords=ChunkIndices([3]) byte_range=From(0)
  |              at icechunk/src/session.rs:1217
  |    4: icechunk::store::get
  |            with key="ABI-L2-MCMIPF/pre-2017-10-31/std_dev_reflectance_factor_C09/c/3" byte_range=From(0)
  |              at icechunk

In [53]:
ingest_all_days(
    ic_repo,
    branch="v2-ingest",
    group="ABI-L2-MCMIPF/pre-2017-10-31",     # ← new group
    start=FIRST_LATE_PREOP_FILE,               # ← new lower bound
    end=LAST_PRE_OPERATIONAL_FILE,             # ← new upper bound
    verify_every=1000,
    zarr_async_concurrency=64,
)

In [28]:
ingest_all_days(
    ic_repo,
    branch="v2-ingest",
    group="ABI-L2-MCMIPF/pre-2023-04-19",
    start=FIRST_OPERATIONAL_FILE,
    end=LAST_PRE_CODEC_CHANGE_FILE,
    verify_every=2000,
    zarr_async_concurrency=64,
)

In [29]:
ingest_all_days(
    ic_repo,
    branch="v2-ingest",
    group="ABI-L2-MCMIPF/post-2023-04-19",
    start=CODEC_CHANGE_MARKER,                # the 4th era's start bound
    end=None,
    verify_every=2000,
    zarr_async_concurrency=64,
)

In [30]:
client.shutdown()

## Access result

In [ ]:
import arraylake as al
import xarray as xr
import matplotlib.pyplot as plt


client = al.Client()
client.login()

In [ ]:
%%time
ic_repo = client.get_repo("earthmover-public/goes-16")

In [ ]:
%%time
session = ic_repo.readonly_session("main")
ds = xr.open_zarr(session.store, chunks=None)
ds

### Full-disk RGB image

In [ ]:
ds_for_img = ds[["CMI"]].isel(t=-1).sel(band=[2, 3, 1])

In [ ]:
%%time
ds_for_img.load()

In [ ]:
# Select visible wavelengths
red = ds_for_img["CMI"].sel(band=2)   # Band 2 (0.64 μm)
veggie = ds_slice["CMI"].sel(band=3)  # Band 3 (0.86 μm) Near-IR, a.k.a. "Veggie"
blue = ds_slice["CMI"].sel(band=1)  # Band 1 (0.47 μm)

In [ ]:
%%time
# Create synthetic green
green = 0.45 * red + 0.10 * veggie + 0.45 * blue

# Stack into RGB, apply gamma correction and clip
rgb = xr.concat([red, green, blue], dim="band").clip(0, 1) ** (1 / 2.2)

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(6, 6), facecolor="black")
ax.set_facecolor("black")
rgb.squeeze(drop=True).plot.imshow(ax=ax, rgb="band")
ax.set_title("GOES-16 True Color - April 8, 2024 18:00 UTC", color="white")
ax.axis("off")
plt.tight_layout()
plt.savefig("eclipse_plot.png", dpi=150, facecolor="black")
plt.show()

### Timeseries

Let's plot a timeseries

In [ ]:
from pyproj import CRS, Transformer

# Find NYC's (x, y) in the file's fixed-grid radians using one sample file.
p = ds["goes_imager_projection"]
H = float(p.perspective_point_height)
crs = CRS.from_cf({
  "grid_mapping_name": "geostationary",
  "perspective_point_height": H,
  "longitude_of_projection_origin": float(p.longitude_of_projection_origin),
  "latitude_of_projection_origin": float(p.latitude_of_projection_origin),
  "semi_major_axis": float(p.semi_major_axis),
  "semi_minor_axis": float(p.semi_minor_axis),
  "sweep_angle_axis": str(p.sweep_angle_axis),
})
nyc_x_m, nyc_y_m = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(
  -74.0060, 40.7128
)
nyc_x, nyc_y = nyc_x_m / H, nyc_y_m / H  # radians, matching the file's x/y coords

In [ ]:
ds_nyc = ds.sel(x=nyc_x, y=nyc_y, method="nearest")